### Imports

In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import datetime as datetime

from tqdm import tqdm
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [2]:
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

### Datasets & Parameter

In [3]:
local_dataset_path = '/Users/E2074/local-datasets/customer/loyalty/experiment2/'

start_date = '20250516'
end_date = '20250526'
city = {'Bangalore' : 'loyalty_experiment2_bangalore', 
        'Delhi' : 'loyalty_experiment2_delhi', 
        'Hyderabad' : 'loyalty_experiment2_hyd'}

city_list = ['Bangalore', 'Delhi', 'Hyderabad']

experiment_name = 'loyalty_experiment2'

start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

date_list = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]

print(date_list)

['20250516', '20250517', '20250518', '20250519', '20250520', '20250521', '20250522', '20250523', '20250524', '20250525', '20250526']


#### Orders

In [4]:
all_dfs = []

for city_folder in city_list[1:-1]:
    folder_path = os.path.join(local_dataset_path, city_folder)
    print(folder_path)
    
    for file_name in os.listdir(folder_path): 
        if file_name.endswith('.parquet'): 
            file_path = os.path.join(folder_path, file_name) 
            df = pd.read_parquet(file_path) 
            all_dfs.append(df)

df_base = pd.concat(all_dfs, ignore_index=True)
df_base.drop_duplicates(inplace=True)

/Users/E2074/local-datasets/customer/loyalty/experiment2/Delhi


In [5]:
test_customer_list = df_base[df_base['sample_category'].isin(['test'])]['customer_id'].tolist()

#### Telemetry

In [6]:
def get_customer_engagement():
    base_path = local_dataset_path + 'telemetry-engagement/'
    parquet_files = glob.glob(os.path.join(base_path, '*_telemetry_engagement_data_dump.parquet'))
    df_all = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    df_all = df_all[(df_all['city'].isin(['Delhi'])) & df_all['userid'].isin(test_customer_list)]
    
    return df_all

df_customer_engagement = get_customer_engagement()

df_new_without_orderid = pd.merge(df_customer_engagement[df_customer_engagement['orderid'].isin([''])], 
                  df_base[['yyyymmdd', 'customer_id', 'quarter_hour', 'order_id']], 
                  how='left', 
                  left_on = ['yyyymmdd', 'userid', 'quarter_hour'],
                  right_on = ['yyyymmdd', 'customer_id', 'quarter_hour'] )
df_new_without_orderid['orderid'] = df_new_without_orderid['order_id'] 
df_new_without_orderid = df_new_without_orderid.drop(['customer_id', 'order_id'], axis=1)

df_customer_engagement_combined = pd.concat([df_customer_engagement[~df_customer_engagement['orderid'].isin([''])], df_new_without_orderid], axis=0)

df_customer_engagement_final = df_customer_engagement_combined\
                                    .groupby(['city', 'userid', 'orderid'])\
                                    .agg(dispatch_viewed_cux = ('dispatch_viewed_cux', 'first'),
                                         dispatch_clicked_cux = ('dispatch_clicked_cux', 'first'),
                                         otw_viewed_cux = ('otw_viewed_cux', 'first'),
                                         otw_clicked_cux = ('otw_clicked_cux', 'first')
                                        )\
                                    .reset_index()
df_customer_engagement_final['both_viewed_cux'] = np.where((~df_customer_engagement_final['dispatch_viewed_cux'].isna()) & (~df_customer_engagement_final['otw_viewed_cux'].isna()),
                                                           df_customer_engagement_final['userid'], None)
df_customer_engagement_final.rename(columns={'userid': 'customer_id', 'orderid': 'order_id'}, inplace=True)

#### Frequency

In [7]:
df_frequency = df_customer_engagement_final.groupby(['customer_id']).agg(frequency= ('order_id', 'nunique')).reset_index()

In [155]:
df_frequency

,customer_id,frequency
0,573f28f59b0ffc283676f96a,1
1,573f28f79b0ffc283677076c,1
2,573f28fe9b0ffc283677181f,1
3,573f28fe9b0ffc28367718ec,1
4,573f29099b0ffc2836774375,3
...,...,...
155563,6834140a45fc35e9a20aa8a8,1
155564,6834144a78be454173632ec0,1
155565,6834147aa090a409c8747b97,1
155566,683414dc4eef7774748c65a6,1


#### MarketPlace

In [8]:
df_marketplace = pd.read_parquet(local_dataset_path + 'marketplace' + '/marketplace_tag.parquet')

### Click Time

In [9]:
def get_click_timing():
    base_path = local_dataset_path + 'click-data/'
    parquet_files = glob.glob(os.path.join(base_path, '*click_data.parquet'))
    df_all = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    df_all['time_taken_to_click'] = abs(df_all['time_taken_to_click'])
    return df_all

df_click_timing = get_click_timing()

In [10]:
df_click_timing.head(1)

,yyyymmdd,city,userid,aduuid,render,view,click,time_taken_to_click
0,20250524,Hyderabad,61d7d931dd82c2e77520b2e9,61d7d931dd82c2e77520b2e9_EA3FFF2B-2F0F-4B7D-95...,1.748053e+09,1.748053e+09,1748052854,106.0


In [11]:
quantiles = df_click_timing['time_taken_to_click'].quantile([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
quantiles

0.1      6.0
0.2     14.0
0.3     29.0
0.4     49.0
0.5     73.0
0.6    104.0
0.7    145.0
0.8    199.0
0.9    290.0
Name: time_taken_to_click, dtype: float64

### Preperiod

In [109]:
local_dataset_path

'/Users/E2074/local-datasets/customer/loyalty/experiment2/'

In [115]:
all_dfs = []

for city_folder in city_list[1:-1]:
    folder_path = os.path.join('/Users/E2074/local-datasets/customer/loyalty/experiment2/Delhi/preperiod')
    print(folder_path)
    
    for file_name in os.listdir(folder_path): 
        if file_name.endswith('.parquet'): 
            file_path = os.path.join(folder_path, file_name) 
            df = pd.read_parquet(file_path) 
            all_dfs.append(df)


df_preperiod = pd.concat(all_dfs, ignore_index=True)
df_preperiod = df_preperiod[df_preperiod['customer_id'].isin(test_customer_list)]
df_preperiod.drop_duplicates(inplace=True)

/Users/E2074/local-datasets/customer/loyalty/experiment2/Delhi/preperiod


In [116]:
df_preperiod['date_type'] = 'pre-period'

In [117]:
df_preperiod.head(1)

,sample_category,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,pickup_location_hex_12,drop_location_hex_12,pickup_location_hex_8,drop_location_hex_8,channel_host,time_bucket,date_type
0,test,20250515,638b5719f470701c3f8ad582,Delhi,Bike Lite,6825f7a7ea13344195ff271b,OCARA,1.747319e+12,1.747319e+12,1.747319e+12,0.0,NaN,278.0,199.0,1747318695281,1945,194815,9.8,0.468,8c3da115056dbff,8c3da10287737ff,883da11501fffff,883da10287fffff,android,evening_peak,pre-period


## Raw Data

In [12]:
print(df_base.shape)
print(df_base.order_id.nunique())
df_base.head(1)

(7835049, 32)
7835049


,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,pickup_location_hex_12,drop_location_hex_12,pickup_location_hex_8,drop_location_hex_8,channel_host,time_bucket
0,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1747882233633,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,0.424,8c3da1118d225ff,8c3da1034d4d5ff,883da1118dfffff,883da1034dfffff,android,morning_peak


In [13]:
print(df_customer_engagement_final.shape)
print(df_customer_engagement_final.order_id.nunique())
df_customer_engagement_final.head(1)

(405556, 8)
405556


,city,customer_id,order_id,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,both_viewed_cux
0,Delhi,573f28f59b0ffc283676f96a,682d71a9e1cc6804ce590db2,573f28f59b0ffc283676f96a,None,573f28f59b0ffc283676f96a,None,573f28f59b0ffc283676f96a


In [14]:
df_frequency.head(1)

,customer_id,frequency
0,573f28f59b0ffc283676f96a,1


In [15]:
df_marketplace.head(1)

,city_name,pickup_location_hex_8,time_bucket,gross_orders,accepted_orders,aor,mp_tag
0,Bangalore,88618928d9fffff,rest_morning,141,65,46.1,3. Poor


In [16]:
df_merge = pd.merge(df_base,
                    df_customer_engagement_final, 
                    how='left', 
                    on=['customer_id', 'order_id'])
df_merge = pd.merge(df_merge,
                    df_frequency, 
                    how='left', 
                    on=['customer_id'])
df_merge = pd.merge(df_merge,
                    df_marketplace, 
                    how='left', 
                    on=['city_name', 'pickup_location_hex_8', 'time_bucket'])

print(df_base.shape)
print(df_base.order_id.nunique())
print(df_merge.shape)
print(df_merge.order_id.nunique())

(7835049, 32)
7835049
(7835049, 43)
7835049


In [17]:
df_merge.columns

Index(['experiment_name', 'execution_name', 'sample_category',
       'taxi_ltr_segment', 'taxi_retention_segments',
       'taxi_regularity_segment', 'taxi_income_segment', 'taxi_intent_segment',
       'yyyymmdd', 'customer_id', 'city_name', 'service_name', 'order_id',
       'modified_order_status', 'customer_cancelled_epoch',
       'order_requested_epoch', 'accepted_epoch', 'accept_to_cancelled',
       'cobra_ttc', 'ocara_ttc', 'tta', 'epoch', 'quarter_hour', 'hhmmss',
       'distance_final_distance', 'accept_to_pickup_distance',
       'pickup_location_hex_12', 'drop_location_hex_12',
       'pickup_location_hex_8', 'drop_location_hex_8', 'channel_host',
       'time_bucket', 'city', 'dispatch_viewed_cux', 'dispatch_clicked_cux',
       'otw_viewed_cux', 'otw_clicked_cux', 'both_viewed_cux', 'frequency',
       'gross_orders', 'accepted_orders', 'aor', 'mp_tag'],
      dtype='object')

In [18]:
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,pickup_location_hex_12,drop_location_hex_12,pickup_location_hex_8,drop_location_hex_8,channel_host,time_bucket,city,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,both_viewed_cux,frequency,gross_orders,accepted_orders,aor,mp_tag
0,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1747882233633,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,0.424,8c3da1118d225ff,8c3da1034d4d5ff,883da1118dfffff,883da1034dfffff,android,morning_peak,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2783.0,2394.0,86.02,1. Good


In [19]:
# df_merge.to_parquet('/Users/E2074/local-datasets/customer/loyalty/experiment2/processed/delhi_merge_file .parquet')

In [19]:
df_merge.to_parquet('/Users/E2074/local-datasets/customer/loyalty/experiment2/processed/delhi_merge_file .parquet')

#### Columns

In [20]:
# First Mile Bins
fm_conds = [
    df_merge['accept_to_pickup_distance'].isna(),
    (df_merge['accept_to_pickup_distance'] >= 0) & (df_merge['accept_to_pickup_distance'] < 0.5),
    (df_merge['accept_to_pickup_distance'] >= 0.5) & (df_merge['accept_to_pickup_distance'] < 1),
    (df_merge['accept_to_pickup_distance'] >= 1) & (df_merge['accept_to_pickup_distance'] < 2),
    (df_merge['accept_to_pickup_distance'] >= 2)
]

fm_choices = ['NA', '0-500m', '0.5-1km', '1-2km', '2+km']
df_merge['fm_bin'] = np.select(fm_conds, fm_choices, default='NA')

# Last Mile Bins
lm_conds = [
    df_merge['distance_final_distance'].isna(),
    (df_merge['distance_final_distance'] >= 0) & (df_merge['distance_final_distance'] < 3),
    (df_merge['distance_final_distance'] >= 3) & (df_merge['distance_final_distance'] < 6),
    (df_merge['distance_final_distance'] >= 6) & (df_merge['distance_final_distance'] < 9),
    (df_merge['distance_final_distance'] >= 9)
]
lm_choices = ['NA', '0-3km', '3-6km', '6-9km', '9+km']
df_merge['lm_bin'] = np.select(lm_conds, lm_choices, default='NA')

# True Test Tag
true_test_cols = ['dispatch_viewed_cux', 'dispatch_clicked_cux', 'otw_viewed_cux', 'otw_clicked_cux', 'both_viewed_cux']
for col in true_test_cols:
    tag_col = f'{col}_tag'
    df_merge[tag_col] = 'test'  # default
    df_merge.loc[df_merge['sample_category'] == 'control', tag_col] = 'control'
    df_merge.loc[(df_merge['sample_category'] == 'test') & df_merge[col].notnull(), tag_col] = 'true test'


# Hex 8 Route
df_merge['hex_8_route'] = df_merge['pickup_location_hex_8'] + ' - ' + df_merge['drop_location_hex_8']

# Updated Segment
df_merge['clicked'] = np.where((~df_merge['dispatch_clicked_cux'].isna()) & (~df_merge['otw_clicked_cux'].isna()), 'yes', None)
df_merge['both_view'] = np.where((~df_merge['both_viewed_cux'].isna()), 'yes', None)
df_merge['single_view'] = np.where((~df_merge['dispatch_viewed_cux'].isna()) | (~df_merge['otw_viewed_cux'].isna()), 'yes', None)

df_merge['updated_segment'] = np.where(df_merge['sample_category'] == 'control', '1. control',
                                np.where((df_merge['clicked'] == 'yes'), '5. clicked', 
                                np.where((df_merge['both_view'] == 'yes'), '4. both_viewed',
                                np.where((df_merge['single_view'] != 'yes'), '3. viewed', '2. not viewed'))))

In [159]:
# frequency

fq_conds = [
    df_merge['frequency'].isna(),
    (df_merge['frequency'] >= 1) & (df_merge['frequency'] < 3),
    (df_merge['frequency'] >= 3) & (df_merge['frequency'] < 5),
    (df_merge['frequency'] >= 5) & (df_merge['frequency'] < 8),
    (df_merge['frequency'] >= 8)
]

fq_choices = ['NA', '1-3', '3-4', '5-7', '8+']
df_merge['fq_bin'] = np.select(fq_conds, fq_choices, default='NA')

In [98]:
df_cux_orders = df_merge.groupby(['customer_id']).agg(orders_per_customers=('order_id', 'nunique')).reset_index()
df_cux_orders

df_merge = pd.merge(df_merge, df_cux_orders, how='left', on=['customer_id'])

order_conds = [
    (df_merge['orders_per_customers'] >= 1) & (df_merge['distance_final_distance'] < 3),
    (df_merge['orders_per_customers'] >= 3) & (df_merge['distance_final_distance'] < 5),
    (df_merge['orders_per_customers'] >= 5) & (df_merge['distance_final_distance'] < 8),
    (df_merge['orders_per_customers'] >= 8)
]
order_choices = ['1-2', '3-4', '5-8', '8+']
df_merge['order_bin'] = np.select(order_conds, order_choices, default='NA')

In [21]:
df_merge.groupby(['sample_category', 'updated_segment'],  dropna=False).agg(cus=('customer_id', 'nunique')).reset_index()

,sample_category,updated_segment,cus
0,control,1. control,1933706
1,test,2. not viewed,101497
2,test,3. viewed,143676
3,test,4. both_viewed,110497
4,test,5. clicked,57


#### User-Defined Functions

In [22]:
def get_funnel(df, grouper):
    start = time.time()
    df_analysis =  df\
    .groupby(grouper, dropna=False)\
    .agg(gross_customer = ('customer_id', 'nunique'),
         gross_orders = ('order_id', 'nunique'),
         net_orders = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'dropped'].nunique()),
         cobra = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'COBRA'].nunique()),
         ocara = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'OCARA'].nunique()),
         cobrm = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'COBRM'].nunique()),
         expired = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'expired'].nunique()),
         aborted = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'aborted'].nunique()),
         
         cobra_ttc_mean = ('cobra_ttc', 'mean'),
         cobra_ttc_p25=('cobra_ttc', lambda x: x.quantile(0.25)),
         cobra_ttc_p50 = ('cobra_ttc', 'median'),
         cobra_ttc_p75=('cobra_ttc', lambda x: x.quantile(0.75)),
         cobra_ttc_p90=('cobra_ttc', lambda x: x.quantile(0.90)),
         # cobra_ttc_p95=('cobra_ttc', lambda x: x.quantile(0.95)),
    
         ocara_ttc_mean = ('ocara_ttc', 'mean'),
         ocara_ttc_p25=('ocara_ttc', lambda x: x.quantile(0.25)),
         ocara_ttc_p50 = ('ocara_ttc', 'median'),
         ocara_ttc_p75=('ocara_ttc', lambda x: x.quantile(0.75)),
         ocara_ttc_p90=('ocara_ttc', lambda x: x.quantile(0.90)),
         # ocara_ttc_p95=('ocara_ttc', lambda x: x.quantile(0.95)),

         tta_mean = ('tta', 'mean'),
         tta_p25=('tta', lambda x: x.quantile(0.25)),
         tta_p50=('tta', 'median'),
         tta_p75=('tta', lambda x: x.quantile(0.75)),
         tta_p90=('tta', lambda x: x.quantile(0.90)),
         # tta_p95=('tta', lambda x: x.quantile(0.95)),


         
        )\
    .reset_index()

    df_analysis['cobra_ttc_mean'] = (df_analysis['cobra_ttc_mean']).round(1)
    df_analysis['ocara_ttc_mean'] = (df_analysis['ocara_ttc_mean']).round(1)
    df_analysis['tta_mean'] = (df_analysis['tta_mean']).round(1)
    
    df_analysis['g2n %'] = (df_analysis['net_orders']*100.00/df_analysis['gross_orders']).round(2)
    
    df_analysis['cobra %'] = (df_analysis['cobra']*100.00/df_analysis['gross_orders']).round(2)
    df_analysis['ocara %'] = (df_analysis['ocara']*100.00/df_analysis['gross_orders']).round(2)
    df_analysis['cobrm %'] = (df_analysis['cobrm']*100.00/df_analysis['gross_orders']).round(2)
    df_analysis['expired %'] = ((df_analysis['expired'] + df_analysis['aborted'] )*100.00/df_analysis['gross_orders']).round(2)


    columns = grouper + ['gross_customer', 'gross_orders', 'net_orders', 'g2n %', 'cobra %', 'ocara %', 'cobrm %', 'expired %',
                         'cobra_ttc_mean', 'ocara_ttc_mean', 'tta_mean',
                         'cobra_ttc_p25', 'cobra_ttc_p50', 'cobra_ttc_p75', 'cobra_ttc_p90', # 'cobra_ttc_p95', 
                         'ocara_ttc_p25', 'ocara_ttc_p50', 'ocara_ttc_p75', 'ocara_ttc_p90', # 'ocara_ttc_p95', 
                         'tta_p25', 'tta_p50', 'tta_p75', 'tta_p90', # 'tta_p95', 
                         # 'cobrm', 'cobra', 'ocara', 'expired', 'aborted'
                        ]
    end = time.time()
    print(f"Execution time: {end - start:.3f} seconds")
    return df_analysis[columns]

## Analysis

In [23]:
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,channel_host,time_bucket,city,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,both_viewed_cux,frequency,gross_orders,accepted_orders,aor,mp_tag,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_viewed_cux_tag,hex_8_route,clicked,both_view,single_view,updated_segment
0,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1747882233633,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,...,android,morning_peak,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2783.0,2394.0,86.02,1. Good,0-500m,6-9km,control,control,control,control,control,883da1118dfffff - 883da1034dfffff,None,None,None,1. control


### City

In [24]:
df_city = get_funnel(df_merge, ['city_name'])
df_city_group_tc = get_funnel(df_merge, ['city_name','sample_category'])
df_city_group_tc_dispatch_view = get_funnel(df_merge, ['city_name','sample_category', 'dispatch_viewed_cux_tag'])
df_city_group_tc_otw_view = get_funnel(df_merge, ['city_name','sample_category', 'otw_viewed_cux_tag'])
df_city_group_tc_both_view = get_funnel(df_merge, ['city_name','sample_category', 'both_viewed_cux_tag'])
df_city_group_tc_updated_seg = get_funnel(df_merge, ['city_name','sample_category', 'updated_segment'])

Execution time: 20.501 seconds
Execution time: 27.168 seconds
Execution time: 23.091 seconds
Execution time: 26.024 seconds
Execution time: 25.185 seconds
Execution time: 26.827 seconds


In [25]:
display(df_city)
display(df_city_group_tc)
display(df_city_group_tc_dispatch_view)
display(df_city_group_tc_otw_view)
display(df_city_group_tc_both_view)
display(df_city_group_tc_updated_seg)

,city_name,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,2147980,7835049,4773453,60.92,15.97,16.89,0.78,5.43,128.9,277.8,68.7,52.0,103.0,181.0,274.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0


,city_name,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.4,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,214274,780418,472108,60.49,15.87,17.16,0.77,5.7,126.6,273.6,68.7,49.0,100.0,178.0,274.0,49.0,154.0,365.0,645.0,6.0,11.0,59.0,173.0


,city_name,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,test,161664,474281,273049,57.57,17.23,18.59,0.81,5.80,121.9,269.3,67.7,48.0,97.0,172.0,263.0,48.0,151.0,358.0,632.0,6.0,11.0,58.0,167.0
2,Delhi,test,true test,141158,306137,199059,65.02,13.77,14.95,0.71,5.55,135.6,282.0,70.2,52.0,108.0,192.0,296.0,50.0,160.0,381.0,664.0,6.0,11.0,60.0,181.0


,city_name,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,test,168889,518220,269030,51.91,22.65,16.52,1.13,7.78,124.7,259.4,67.0,48.0,98.0,176.0,271.0,43.0,143.0,345.0,619.0,6.0,11.0,59.0,168.0
2,Delhi,test,true test,131443,262198,203078,77.45,2.47,18.44,0.05,1.59,160.0,298.9,71.1,72.0,136.0,223.0,325.0,59.0,176.0,400.0,685.8,6.0,11.0,59.0,180.0


,city_name,sample_category,both_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.00,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,test,181650,604996,337364,55.76,19.65,16.78,0.97,6.84,125.2,264.9,67.0,49.0,99.0,177.0,272.0,46.0,148.0,353.00,626.0,6.0,11.0,58.0,168.0
2,Delhi,test,true test,110510,175422,134744,76.81,2.86,18.49,0.06,1.78,158.6,301.1,73.1,72.0,135.0,220.0,322.0,58.0,176.0,403.25,697.0,6.0,10.0,59.0,187.0


,city_name,sample_category,updated_segment,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,1. control,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.00,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,2. not viewed,101497,217467,132635,60.99,17.75,13.45,0.96,6.85,133.8,267.7,65.6,51.0,106.0,189.75,295.0,46.0,152.0,360.0,629.0,6.0,11.0,59.0,169.0
2,Delhi,test,3. viewed,143676,387501,204713,52.83,20.71,18.65,0.98,6.83,121.1,263.7,67.8,47.0,96.0,171.00,262.0,46.0,146.0,349.0,624.0,6.0,11.0,58.0,167.0
3,Delhi,test,4. both_viewed,110497,175387,134717,76.81,2.86,18.49,0.06,1.78,158.6,301.0,73.1,72.0,135.0,220.00,322.0,58.0,176.0,403.0,697.0,6.0,10.0,59.0,187.0
4,Delhi,test,5. clicked,57,63,43,68.25,7.94,17.46,0.00,6.35,121.8,485.0,226.6,104.0,118.0,152.00,166.4,166.5,546.0,630.5,660.0,11.0,80.0,244.0,586.8


### Time Bucket

In [92]:
df_city_timebucket = get_funnel(df_merge[df_merge['time_bucket'].isin(['afternoon', 'morning_peak', 'evening_peak'])], ['city_name', 'time_bucket'])
df_city_timebucket_group_tc = get_funnel(df_merge[df_merge['time_bucket'].isin(['afternoon', 'morning_peak', 'evening_peak'])], ['city_name', 'time_bucket', 'sample_category'])
df_city_timebucket_group_tc_dispatch_view = get_funnel(df_merge[df_merge['time_bucket'].isin(['afternoon', 'morning_peak', 'evening_peak'])], ['city_name', 'time_bucket', 'sample_category', 'dispatch_viewed_cux_tag'])
df_city_timebucket_group_tc_otw_view = get_funnel(df_merge[df_merge['time_bucket'].isin(['afternoon', 'morning_peak', 'evening_peak'])], ['city_name', 'time_bucket', 'sample_category', 'otw_viewed_cux_tag'])
df_city_timebucket_group_tc_both_view = get_funnel(df_merge[df_merge['time_bucket'].isin(['afternoon', 'morning_peak', 'evening_peak'])], ['city_name', 'time_bucket', 'sample_category', 'both_viewed_cux_tag'])
df_city_timebucket_group_tc_updated_seg = get_funnel(df_merge[df_merge['time_bucket'].isin(['afternoon', 'morning_peak', 'evening_peak'])], ['city_name', 'time_bucket', 'sample_category', 'updated_segment'])

Execution time: 24.823 seconds
Execution time: 26.494 seconds
Execution time: 27.916 seconds
Execution time: 28.343 seconds
Execution time: 27.021 seconds
Execution time: 28.671 seconds


In [93]:
display(df_city_timebucket)
display(df_city_timebucket_group_tc)
display(df_city_timebucket_group_tc_dispatch_view)
display(df_city_timebucket_group_tc_otw_view)
display(df_city_timebucket_group_tc_both_view)
display(df_city_timebucket_group_tc_updated_seg)

,city_name,time_bucket,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,afternoon,955108,2001600,1260973,63.00,14.23,17.06,0.77,4.94,120.1,294.6,67.7,48.0,94.0,168.0,259.0,57.0,171.0,394.0,683.0,6.0,11.0,55.0,161.0
1,Delhi,evening_peak,1036822,2489612,1500384,60.27,15.85,18.01,0.58,5.29,122.5,275.0,69.0,50.0,98.0,172.0,261.0,50.0,154.0,360.0,642.0,6.0,12.0,60.0,169.0
2,Delhi,morning_peak,870277,2178904,1350749,61.99,17.76,14.52,0.49,5.24,139.8,254.8,69.0,58.0,114.0,197.0,292.0,48.0,146.0,346.0,607.0,6.0,11.0,66.0,184.0


,city_name,time_bucket,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,afternoon,control,860161,1802581,1136728,63.06,14.24,17.03,0.77,4.89,120.2,295.0,67.7,49.0,94.0,168.0,258.0,57.0,172.0,394.0,683.0,6.0,11.0,55.0,161.0
1,Delhi,afternoon,test,94947,199019,124245,62.43,14.13,17.32,0.77,5.35,118.8,291.6,67.5,45.0,92.0,167.0,263.0,54.0,167.0,391.0,682.0,6.0,11.0,55.0,163.0
2,Delhi,evening_peak,control,933410,2241265,1352003,60.32,15.81,17.99,0.58,5.31,122.9,275.5,69.1,50.0,98.0,172.0,261.0,51.0,155.0,361.0,642.0,6.0,12.0,60.0,169.0
3,Delhi,evening_peak,test,103412,248347,148381,59.75,16.23,18.28,0.59,5.16,118.8,271.3,68.0,47.0,95.0,168.0,255.0,48.0,152.0,358.0,636.0,6.0,11.0,60.0,168.0
4,Delhi,morning_peak,control,783656,1961853,1216493,62.01,17.82,14.49,0.50,5.19,139.9,255.5,68.9,59.0,114.0,197.0,291.0,48.0,146.0,347.0,608.0,6.0,11.0,66.0,183.0
5,Delhi,morning_peak,test,86621,217051,134256,61.85,17.19,14.81,0.45,5.69,138.7,249.0,69.8,55.0,112.0,196.0,300.0,44.0,140.0,339.0,596.8,6.0,11.0,67.0,189.0


,city_name,time_bucket,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,afternoon,control,control,860161,1802581,1136728,63.06,14.24,17.03,0.77,4.89,120.2,295.0,67.7,49.00,94.0,168.00,258.0,57.0,172.0,394.00,683.0,6.0,11.0,55.0,161.0
1,Delhi,afternoon,test,test,60137,109360,67236,61.48,15.11,17.34,0.84,5.23,116.8,279.1,63.1,45.00,91.0,163.00,256.2,52.0,160.0,374.00,658.0,6.0,10.0,53.0,152.0
2,Delhi,afternoon,test,true test,59077,89659,57009,63.58,12.93,17.30,0.69,5.49,121.5,306.8,72.8,45.00,93.0,171.00,274.0,55.0,175.0,408.00,715.0,6.0,11.0,57.0,179.0
3,Delhi,evening_peak,control,control,933410,2241265,1352003,60.32,15.81,17.99,0.58,5.31,122.9,275.5,69.1,50.00,98.0,172.00,261.0,51.0,155.0,361.00,642.0,6.0,12.0,60.0,169.0
4,Delhi,evening_peak,test,test,99756,224553,125426,55.86,17.82,20.04,0.64,5.64,118.7,271.1,69.8,47.00,95.0,168.00,254.0,48.0,151.0,358.00,636.0,6.0,11.0,60.0,171.0
5,Delhi,evening_peak,test,true test,16960,23794,22955,96.47,1.18,1.61,0.10,0.64,132.9,301.7,55.2,49.75,100.5,175.25,317.1,50.0,171.0,382.75,665.9,6.0,12.0,59.0,151.0
6,Delhi,morning_peak,control,control,783656,1961853,1216493,62.01,17.82,14.49,0.50,5.19,139.9,255.5,68.9,59.00,114.0,197.00,291.0,48.0,146.0,347.00,608.0,6.0,11.0,66.0,183.0
7,Delhi,morning_peak,test,test,38954,78340,47711,60.90,18.43,14.18,0.51,5.98,135.6,230.9,64.4,53.00,107.0,191.00,296.0,41.0,127.0,311.00,549.0,6.0,11.0,63.0,176.0
8,Delhi,morning_peak,test,true test,74516,138711,86545,62.39,16.49,15.16,0.42,5.53,140.7,258.6,72.8,56.00,115.0,199.00,302.0,46.0,147.0,353.00,623.0,6.0,11.0,69.0,196.0


,city_name,time_bucket,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,afternoon,control,control,860161,1802581,1136728,63.06,14.24,17.03,0.77,4.89,120.2,295.0,67.7,49.0,94.0,168.0,258.0,57.0,172.0,394.0,683.0,6.0,11.0,55.0,161.0
1,Delhi,afternoon,test,test,54408,99642,46875,47.04,27.32,14.47,1.51,9.65,119.5,254.9,60.4,45.0,93.0,168.0,264.0,40.0,139.0,345.0,617.0,6.0,10.0,50.0,146.0
2,Delhi,afternoon,test,true test,66742,99377,77370,77.86,0.90,20.17,0.03,1.03,96.9,318.0,72.0,33.0,68.0,129.5,220.8,66.0,189.0,420.0,721.0,6.0,11.0,57.0,176.0
3,Delhi,evening_peak,control,control,933410,2241265,1352003,60.32,15.81,17.99,0.58,5.31,122.9,275.5,69.1,50.0,98.0,172.0,261.0,51.0,155.0,361.0,642.0,6.0,12.0,60.0,169.0
4,Delhi,evening_peak,test,test,100334,236999,138854,58.59,16.93,18.53,0.62,5.34,118.6,270.1,68.5,47.0,95.0,168.0,254.0,47.0,150.0,355.0,634.0,6.0,12.0,60.0,169.0
5,Delhi,evening_peak,test,true test,9945,11348,9527,83.95,1.59,13.02,0.05,1.38,160.4,308.6,60.8,77.0,142.0,234.0,292.0,64.0,183.0,431.0,734.0,6.0,11.0,52.0,154.0
6,Delhi,morning_peak,control,control,783656,1961853,1216493,62.01,17.82,14.49,0.50,5.19,139.9,255.5,68.9,59.0,114.0,197.0,291.0,48.0,146.0,347.0,608.0,6.0,11.0,66.0,183.0
7,Delhi,morning_peak,test,test,49976,103447,47193,45.62,31.11,12.84,0.89,9.54,133.2,219.1,66.9,52.0,105.0,188.0,292.0,35.0,116.0,297.0,538.0,6.0,11.0,69.0,185.0
8,Delhi,morning_peak,test,true test,65855,113604,87063,76.64,4.51,16.60,0.06,2.19,173.5,270.0,71.5,90.0,151.0,238.0,339.0,52.0,157.0,368.0,633.0,6.0,11.0,66.0,191.0


,city_name,time_bucket,sample_category,both_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,afternoon,control,control,860161,1802581,1136728,63.06,14.24,17.03,0.77,4.89,120.2,295.0,67.7,49.0,94.0,168.00,258.0,57.0,172.0,394.0,683.0,6.0,11.0,55.0,161.0
1,Delhi,afternoon,test,test,70516,138139,77015,55.75,19.91,16.04,1.10,7.20,119.3,273.2,62.9,45.0,92.0,167.00,264.0,48.0,155.0,369.0,646.0,6.0,10.0,53.0,153.0
2,Delhi,afternoon,test,true test,47957,60880,47230,77.58,1.02,20.22,0.03,1.15,97.5,324.7,75.1,33.0,70.0,132.25,218.4,64.0,192.0,429.0,738.0,6.0,11.0,58.0,184.0
3,Delhi,evening_peak,control,control,933410,2241265,1352003,60.32,15.81,17.99,0.58,5.31,122.9,275.5,69.1,50.0,98.0,172.00,261.0,51.0,155.0,361.0,642.0,6.0,12.0,60.0,169.0
4,Delhi,evening_peak,test,test,102553,244073,144420,59.17,16.49,18.50,0.60,5.24,118.8,271.3,68.2,47.0,95.0,168.00,255.0,48.0,151.0,358.0,636.0,6.0,11.0,60.0,168.0
5,Delhi,evening_peak,test,true test,3776,4274,3961,92.68,1.15,5.31,0.02,0.84,145.8,278.3,58.2,49.0,116.0,220.00,317.4,51.0,189.0,395.5,656.8,6.0,11.0,57.0,163.3
6,Delhi,morning_peak,control,control,783656,1961853,1216493,62.01,17.82,14.49,0.50,5.19,139.9,255.5,68.9,59.0,114.0,197.00,291.0,48.0,146.0,347.0,608.0,6.0,11.0,66.0,183.0
7,Delhi,morning_peak,test,test,58459,135111,72413,53.60,24.58,13.48,0.69,7.66,134.8,229.0,66.9,53.0,107.0,191.00,295.0,39.0,125.0,310.0,551.0,6.0,11.0,67.0,184.0
8,Delhi,morning_peak,test,true test,56860,81940,61843,75.47,5.01,17.00,0.06,2.45,170.6,275.2,73.3,88.0,149.0,233.00,334.0,53.0,158.0,377.0,649.0,6.0,11.0,66.0,195.0


,city_name,time_bucket,sample_category,updated_segment,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,afternoon,control,1. control,860161,1802581,1136728,63.06,14.24,17.03,0.77,4.89,120.2,295.0,67.7,49.0,94.0,168.00,258.0,57.00,172.0,394.00,683.0,6.0,11.0,55.00,161.0
1,Delhi,afternoon,test,2. not viewed,45078,67270,39916,59.34,16.72,16.26,0.92,6.77,122.2,287.0,65.8,45.0,94.0,173.00,275.0,52.00,166.5,385.00,668.0,6.0,11.0,56.00,161.0
2,Delhi,afternoon,test,3. viewed,41677,70863,37096,52.35,22.94,15.83,1.27,7.61,117.2,259.7,59.9,45.0,91.0,164.00,257.0,45.00,143.0,350.00,623.0,6.0,10.0,48.00,143.0
3,Delhi,afternoon,test,4. both_viewed,47952,60869,47220,77.58,1.03,20.22,0.03,1.15,97.5,324.7,75.1,33.0,70.0,132.25,218.4,64.00,192.0,429.00,738.0,6.0,11.0,58.00,184.0
4,Delhi,afternoon,test,5. clicked,16,17,13,76.47,5.88,11.76,0.00,5.88,176.0,403.5,290.5,176.0,176.0,176.00,176.0,275.25,403.5,531.75,608.7,59.0,205.0,360.25,653.0
5,Delhi,evening_peak,control,1. control,933410,2241265,1352003,60.32,15.81,17.99,0.58,5.31,122.9,275.5,69.1,50.0,98.0,172.00,261.0,51.00,155.0,361.00,642.0,6.0,12.0,60.00,169.0
6,Delhi,evening_peak,test,2. not viewed,19560,26594,24560,92.35,1.36,5.29,0.10,0.89,143.1,316.6,56.6,60.0,116.0,193.00,299.8,64.00,179.0,432.00,743.8,6.0,11.0,57.00,148.0
7,Delhi,evening_peak,test,3. viewed,97442,217478,119859,55.11,18.34,20.12,0.66,5.77,118.6,269.8,70.1,47.0,95.0,168.00,254.0,47.00,150.0,355.00,634.0,6.0,11.0,60.00,172.0
8,Delhi,evening_peak,test,4. both_viewed,3775,4272,3959,92.67,1.15,5.31,0.02,0.84,145.8,278.3,58.2,49.0,116.0,220.00,317.4,51.00,189.0,395.50,656.8,6.0,11.0,57.00,163.5
9,Delhi,evening_peak,test,5. clicked,3,3,3,100.00,0.00,0.00,0.00,0.00,NaN,NaN,6.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,7.0,7.50,7.8


### MarketPlace Tag

In [86]:
df_city_mp_tag = get_funnel(df_merge, ['city_name', 'mp_tag'])
df_city_mp_tag_group_tc = get_funnel(df_merge, ['city_name', 'mp_tag', 'sample_category'])
df_city_mp_tag_group_tc_dispatch_view = get_funnel(df_merge, ['city_name', 'mp_tag', 'sample_category', 'dispatch_viewed_cux_tag'])
df_city_mp_tag_group_tc_otw_view = get_funnel(df_merge, ['city_name', 'mp_tag', 'sample_category', 'otw_viewed_cux_tag'])
df_city_mp_tag_group_tc_both_view = get_funnel(df_merge, ['city_name', 'mp_tag', 'sample_category', 'both_viewed_cux_tag'])
df_city_mp_tag_group_tc_updated_seg = get_funnel(df_merge, ['city_name', 'mp_tag', 'sample_category', 'updated_segment'])

Execution time: 24.791 seconds
Execution time: 25.729 seconds
Execution time: 26.804 seconds
Execution time: 26.741 seconds
Execution time: 27.284 seconds
Execution time: 27.302 seconds


In [87]:
display(df_city_mp_tag)
display(df_city_mp_tag_group_tc)
display(df_city_mp_tag_group_tc_dispatch_view)
display(df_city_mp_tag_group_tc_otw_view)
display(df_city_mp_tag_group_tc_both_view)
display(df_city_mp_tag_group_tc_updated_seg)

,city_name,mp_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,1387192,3505349,2364847,67.46,11.18,17.49,0.43,3.44,113.8,278.2,61.1,46.0,89.0,157.0,245.0,51.0,158.0,370.0,648.0,6.0,9.0,46.0,139.0
1,Delhi,2. Balance,1101913,2734479,1642256,60.06,16.82,17.17,0.65,5.31,126.1,278.5,71.8,52.0,101.0,176.0,266.0,51.0,159.0,372.0,656.0,6.0,12.0,64.0,180.0
2,Delhi,3. Poor,562035,1548945,754211,48.69,25.20,15.10,1.41,9.60,146.8,271.8,84.5,62.0,121.0,208.0,302.0,54.0,158.0,362.0,635.0,6.0,16.0,95.0,230.0
3,Delhi,NaN,23541,46276,12139,26.23,19.62,15.36,15.04,23.74,153.7,394.2,109.8,57.0,118.0,224.0,335.0,75.0,226.0,528.0,923.1,6.0,11.0,78.0,296.0


,city_name,mp_tag,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.0,89.0,157.0,245.0,51.00,159.0,371.0,649.0,6.0,9.0,46.00,139.0
1,Delhi,1. Good,test,138274,349090,233697,66.94,11.30,17.85,0.41,3.50,111.2,271.1,60.4,43.0,87.0,154.0,245.0,47.00,153.0,365.0,639.0,6.0,9.0,45.00,138.0
2,Delhi,2. Balance,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.0,101.0,176.0,266.0,52.00,159.0,372.0,656.0,6.0,12.0,64.00,179.0
3,Delhi,2. Balance,test,109970,272148,162656,59.77,16.63,17.35,0.62,5.63,124.9,276.5,71.8,49.0,100.0,177.0,269.0,49.00,156.0,369.5,655.0,6.0,12.0,64.00,182.0
4,Delhi,3. Poor,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.0,121.0,208.0,302.0,54.00,158.0,363.0,636.0,6.0,16.0,95.00,230.0
5,Delhi,3. Poor,test,56014,154526,74589,48.27,24.74,15.37,1.41,10.22,143.9,271.4,86.0,59.0,117.0,206.0,304.0,51.00,154.0,357.0,629.7,6.0,16.0,98.00,233.0
6,Delhi,NaN,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.0,119.0,224.0,335.0,77.00,230.0,533.0,923.0,6.0,11.0,78.00,295.0
7,Delhi,NaN,test,2327,4654,1166,25.05,20.00,14.91,15.23,24.80,149.3,378.8,110.4,49.0,110.0,223.0,336.0,62.25,187.5,484.5,926.0,6.0,11.0,82.25,301.9


,city_name,mp_tag,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.00,89.0,157.00,245.0,51.0,159.0,371.00,649.0,6.0,9.0,46.0,139.0
1,Delhi,1. Good,test,test,102105,222100,141931,63.90,12.65,19.34,0.41,3.69,107.9,266.3,60.3,42.00,85.0,149.00,235.0,46.0,149.0,358.00,628.0,6.0,9.0,46.0,136.0
2,Delhi,1. Good,test,true test,75575,126990,91766,72.26,8.93,15.24,0.39,3.17,119.2,281.9,60.5,44.00,92.0,165.00,270.0,49.0,162.0,380.00,658.3,5.0,9.0,42.0,140.0
3,Delhi,2. Balance,control,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.00,101.0,176.00,266.0,52.0,159.0,372.00,656.0,6.0,12.0,64.0,179.0
4,Delhi,2. Balance,test,test,77909,165158,93261,56.47,18.25,18.92,0.64,5.73,121.6,271.7,71.3,48.00,98.0,172.00,261.0,49.0,153.0,360.00,640.0,6.0,12.0,64.0,178.0
5,Delhi,2. Balance,test,true test,61294,106990,69395,64.86,14.14,14.92,0.60,5.47,131.5,286.0,72.7,50.00,105.0,186.00,287.0,50.0,162.0,387.00,682.8,6.0,12.0,66.0,189.0
6,Delhi,3. Poor,control,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.00,121.0,208.00,302.0,54.0,158.0,363.00,636.0,6.0,16.0,95.0,230.0
7,Delhi,3. Poor,test,test,35744,84119,37177,44.20,27.15,16.11,1.69,10.85,138.8,269.5,84.3,56.00,113.0,198.00,293.0,52.0,154.0,348.00,618.0,6.0,16.0,95.0,229.0
8,Delhi,3. Poor,test,true test,36033,70407,37412,53.14,21.85,14.48,1.06,9.47,151.5,273.9,87.7,63.00,125.0,215.00,320.0,51.0,154.0,367.00,640.5,6.0,16.0,100.0,238.0
9,Delhi,NaN,control,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.00,119.0,224.00,335.0,77.0,230.0,533.00,923.0,6.0,11.0,78.0,295.0


,city_name,mp_tag,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.00,89.0,157.0,245.0,51.00,159.0,371.0,649.0,6.0,9.0,46.0,139.0
1,Delhi,1. Good,test,test,103130,228723,137048,59.92,16.79,17.74,0.60,4.94,111.1,257.1,59.3,43.00,87.0,154.0,244.0,41.00,141.0,345.0,616.0,6.0,9.0,46.0,136.0
2,Delhi,1. Good,test,true test,74196,120367,96649,80.30,0.86,18.05,0.03,0.76,113.1,297.3,62.1,42.00,89.0,160.0,251.0,60.00,177.0,397.0,679.0,5.0,9.0,42.0,141.0
3,Delhi,2. Balance,control,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.00,101.0,176.0,266.0,52.00,159.0,372.0,656.0,6.0,12.0,64.0,179.0
4,Delhi,2. Balance,test,test,82653,182466,93091,51.02,23.72,16.70,0.90,7.66,124.0,262.6,70.0,49.00,99.0,175.5,268.0,44.00,144.0,347.0,627.0,6.0,12.0,64.0,177.0
5,Delhi,2. Balance,test,true test,56486,89682,69565,77.57,2.21,18.66,0.06,1.50,145.2,301.9,74.4,59.00,124.0,205.0,288.7,59.00,179.0,410.0,701.0,6.0,12.0,64.0,191.0
6,Delhi,3. Poor,control,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.00,121.0,208.0,302.0,54.00,158.0,363.0,636.0,6.0,16.0,95.0,230.0
7,Delhi,3. Poor,test,test,41203,103188,38215,37.03,33.71,13.68,2.06,13.51,140.1,256.1,84.9,56.00,113.0,200.0,297.0,47.25,145.0,337.0,595.0,6.0,16.0,98.0,230.0
8,Delhi,3. Poor,test,true test,31027,51338,36374,70.85,6.70,18.75,0.08,3.61,182.6,293.9,87.2,97.00,157.0,250.0,352.0,58.00,167.0,389.0,667.5,6.0,16.0,97.0,237.0
9,Delhi,NaN,control,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.00,119.0,224.0,335.0,77.00,230.0,533.0,923.0,6.0,11.0,78.0,295.0


,city_name,mp_tag,sample_category,both_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.00,89.0,157.00,245.0,51.0,159.0,371.00,649.0,6.0,9.0,46.0,139.0
1,Delhi,1. Good,test,test,114638,270824,170991,63.14,14.28,17.80,0.51,4.27,111.1,262.2,59.3,43.00,87.0,154.00,245.0,44.0,146.0,353.00,623.6,6.0,9.0,46.0,135.0
2,Delhi,1. Good,test,true test,57048,78266,62706,80.12,1.00,18.01,0.03,0.84,112.7,301.5,63.7,42.75,88.0,160.00,246.2,58.0,178.0,400.00,687.0,5.0,9.0,41.0,147.0
3,Delhi,2. Balance,control,control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.00,101.0,176.00,266.0,52.0,159.0,372.00,656.0,6.0,12.0,64.0,179.0
4,Delhi,2. Balance,test,test,90561,211624,116031,54.83,20.68,16.95,0.78,6.76,124.3,268.2,70.2,49.00,99.0,176.00,269.0,46.0,149.0,356.00,634.0,6.0,12.0,64.0,177.0
5,Delhi,2. Balance,test,true test,44263,60524,46625,77.04,2.49,18.72,0.07,1.68,142.9,302.8,76.2,57.00,123.0,199.75,284.0,60.0,178.0,412.00,712.8,6.0,11.0,65.0,198.0
6,Delhi,3. Poor,control,control,506021,1394419,679622,48.74,25.25,15.07,1.41,9.53,147.1,271.8,84.3,63.00,121.0,208.00,302.0,54.0,158.0,363.00,636.0,6.0,16.0,95.0,230.0
7,Delhi,3. Poor,test,test,44912,118475,49525,41.80,29.97,14.29,1.80,12.13,141.1,262.1,84.4,57.00,115.0,201.00,299.0,49.0,149.0,341.25,609.0,6.0,16.0,97.0,230.0
8,Delhi,3. Poor,test,true test,25342,36051,25064,69.52,7.53,18.91,0.10,3.94,180.3,294.6,89.3,96.00,155.0,248.00,348.0,56.0,167.0,394.25,675.0,6.0,15.0,98.0,240.0
9,Delhi,NaN,control,control,21214,41622,10973,26.36,19.58,15.41,15.02,23.62,154.2,395.9,109.7,58.00,119.0,224.00,335.0,77.0,230.0,533.00,923.0,6.0,11.0,78.0,295.0


,city_name,mp_tag,sample_category,updated_segment,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,1. control,1248918,3156259,2131150,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.00,89.0,157.00,245.0,51.00,159.0,371.00,649.0,6.0,9.0,46.00,139.0
1,Delhi,1. Good,test,2. not viewed,54939,90816,62996,69.37,11.91,14.19,0.53,4.00,119.6,264.8,56.5,44.00,92.0,166.00,272.0,46.00,153.0,362.00,627.0,6.0,9.0,43.00,130.0
2,Delhi,1. Good,test,3. viewed,87679,179998,107988,59.99,15.47,19.62,0.50,4.41,107.8,261.3,60.7,42.00,85.0,149.00,235.0,43.00,144.0,350.00,622.0,6.0,10.0,47.00,138.0
3,Delhi,1. Good,test,4. both_viewed,57042,78254,62696,80.12,1.00,18.01,0.03,0.84,112.7,301.6,63.7,42.50,88.0,160.00,246.4,58.00,178.0,400.00,687.0,5.0,9.0,41.00,147.0
4,Delhi,1. Good,test,5. clicked,20,22,17,77.27,9.09,13.64,0.00,0.00,81.5,205.7,134.2,70.25,81.5,92.75,99.5,133.00,186.0,268.50,318.0,14.0,79.5,231.75,310.5
5,Delhi,2. Balance,control,1. control,991943,2462331,1479600,60.09,16.84,17.15,0.65,5.27,126.2,278.7,71.8,52.00,101.0,176.00,266.0,52.00,159.0,372.00,656.0,6.0,12.0,64.00,179.0
6,Delhi,2. Balance,test,2. not viewed,44156,75617,45708,60.45,18.65,13.27,0.81,6.83,131.0,274.3,67.9,50.00,104.0,185.00,290.0,45.00,155.0,368.50,642.0,6.0,12.0,64.00,176.0
7,Delhi,2. Balance,test,3. viewed,67473,135999,70320,51.71,21.81,19.00,0.76,6.72,121.1,265.8,71.4,48.00,97.0,171.00,260.0,47.00,147.0,351.00,631.0,6.0,12.0,64.00,178.0
8,Delhi,2. Balance,test,4. both_viewed,44257,60512,46615,77.03,2.49,18.73,0.07,1.68,142.9,302.8,76.2,57.00,123.0,199.75,284.0,60.00,178.0,412.00,713.0,6.0,11.0,65.00,198.0
9,Delhi,2. Balance,test,5. clicked,17,20,13,65.00,5.00,15.00,0.00,15.00,176.0,400.3,176.9,176.00,176.0,176.00,176.0,287.50,548.0,587.00,610.4,9.0,134.0,239.00,482.4


### Segment

In [94]:
df_city_segment = get_funnel(df_merge, ['city_name', 'taxi_regularity_segment'])
df_city_segment_group_tc = get_funnel(df_merge, ['city_name', 'taxi_regularity_segment', 'sample_category'])
df_city_segment_group_tc_dispatch_view = get_funnel(df_merge, ['city_name', 'taxi_regularity_segment', 'sample_category', 'dispatch_viewed_cux_tag'])
df_city_segment_group_tc_otw_view = get_funnel(df_merge, ['city_name', 'taxi_regularity_segment', 'sample_category', 'otw_viewed_cux_tag'])
df_city_segment_group_tc_both_view = get_funnel(df_merge, ['city_name', 'taxi_regularity_segment', 'sample_category', 'both_viewed_cux_tag'])
df_city_segment_group_tc_updated_seg = get_funnel(df_merge, ['city_name', 'taxi_regularity_segment', 'sample_category', 'updated_segment'])

Execution time: 25.031 seconds
Execution time: 26.875 seconds
Execution time: 29.815 seconds
Execution time: 29.239 seconds
Execution time: 28.244 seconds
Execution time: 28.125 seconds


In [95]:
display(df_city_segment)
display(df_city_segment_group_tc)
display(df_city_segment_group_tc_dispatch_view)
display(df_city_segment_group_tc_otw_view)
display(df_city_segment_group_tc_both_view)
display(df_city_segment_group_tc_updated_seg)

,city_name,taxi_regularity_segment,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,393427,1142200,697991,61.11,16.07,16.65,0.85,5.33,128.7,298.2,70.3,53.0,103.0,180.0,272.0,60.0,178.0,401.0,688.0,6.0,11.0,58.0,173.0
1,Delhi,DAILY,371116,2913809,1824207,62.61,16.30,15.97,0.55,4.59,127.5,222.3,60.6,51.0,102.0,180.0,272.0,41.0,126.0,300.0,532.0,6.0,11.0,59.0,161.0
2,Delhi,MONTHLY,203950,504222,306684,60.82,15.77,16.74,0.87,5.79,133.0,324.6,75.6,56.0,107.0,185.0,279.0,68.0,197.0,431.0,734.0,6.0,11.0,60.0,183.0
3,Delhi,RARE_NEED,200969,499591,302904,60.63,16.03,16.37,0.85,6.11,134.1,330.2,76.5,56.0,108.0,187.0,281.0,70.0,200.0,439.0,740.0,6.0,11.0,61.0,184.0
4,Delhi,UNKNOWN,279944,603155,340396,56.44,15.00,19.40,1.26,7.89,135.0,368.4,92.5,54.0,108.0,189.0,286.0,75.0,225.0,490.0,825.0,6.0,11.0,65.0,218.0
5,Delhi,WEEKLY,510343,1749430,1058617,60.51,16.48,17.08,0.75,5.17,125.8,260.8,64.0,51.0,100.0,177.0,267.0,50.0,150.0,349.0,615.0,6.0,11.0,56.0,161.0
6,Delhi,NaN,188231,422642,242654,57.41,12.85,20.35,1.53,7.86,134.4,372.7,89.9,53.0,106.0,188.0,287.0,66.0,216.0,493.0,842.0,6.0,9.0,54.0,209.0


,city_name,taxi_regularity_segment,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.0,103.0,180.0,271.0,60.0,178.0,401.0,688.0,6.0,11.0,58.0,172.0
1,Delhi,BI_WEEKLY,test,39421,115497,70006,60.61,15.91,16.88,0.89,5.71,127.1,296.7,70.8,51.0,100.0,177.0,274.0,56.0,175.0,398.0,680.0,6.0,11.0,59.0,177.0
2,Delhi,DAILY,control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.0,102.0,180.0,272.0,41.0,126.0,300.0,532.0,6.0,11.0,59.0,161.0
3,Delhi,DAILY,test,36849,288651,179654,62.24,16.24,16.30,0.50,4.71,125.5,219.5,60.6,48.0,100.0,178.0,273.0,38.0,121.0,298.0,533.0,6.0,11.0,59.0,162.0
4,Delhi,MONTHLY,control,183552,453122,275975,60.91,15.78,16.70,0.87,5.75,133.1,325.8,75.6,56.0,107.0,185.0,279.0,68.0,198.0,431.0,736.0,6.0,11.0,60.0,183.0
5,Delhi,MONTHLY,test,20398,51100,30709,60.10,15.75,17.08,0.95,6.13,131.6,314.0,76.1,53.0,105.0,184.0,279.0,66.0,190.5,426.0,717.5,6.0,11.0,59.0,186.0
6,Delhi,RARE_NEED,control,180887,449883,272826,60.64,16.10,16.36,0.85,6.05,134.3,331.0,76.4,57.0,108.0,188.0,281.0,70.0,200.0,439.0,739.0,6.0,11.0,61.0,184.0
7,Delhi,RARE_NEED,test,20082,49708,30078,60.51,15.47,16.54,0.82,6.66,131.5,323.5,77.1,53.0,106.0,183.0,285.2,72.0,203.0,440.0,749.0,6.0,11.0,62.0,191.0
8,Delhi,UNKNOWN,control,251997,543104,306750,56.48,15.03,19.41,1.26,7.81,135.4,368.0,92.5,55.0,108.0,189.0,286.0,75.0,225.0,490.0,824.0,6.0,11.0,65.0,218.0
9,Delhi,UNKNOWN,test,27947,60051,33646,56.03,14.74,19.34,1.25,8.64,131.0,372.1,92.4,50.0,103.0,183.0,284.0,72.0,224.0,489.0,836.0,6.0,11.0,66.0,222.0


,city_name,taxi_regularity_segment,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.0,103.0,180.0,271.0,60.0,178.0,401.0,688.0,6.0,11.0,58.0,172.0
1,Delhi,BI_WEEKLY,test,test,28909,68976,40302,58.43,17.16,17.77,0.95,5.70,122.5,291.8,69.6,49.0,97.0,172.0,264.0,55.0,172.0,392.0,676.0,6.0,11.0,58.0,170.0
2,Delhi,BI_WEEKLY,test,true test,24869,46521,29704,63.85,14.06,15.57,0.80,5.72,135.5,305.0,72.5,53.0,107.0,190.0,295.0,57.0,180.0,411.0,685.0,6.0,11.0,60.0,186.0
3,Delhi,DAILY,control,control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.0,102.0,180.0,272.0,41.0,126.0,300.0,532.0,6.0,11.0,59.0,161.0
4,Delhi,DAILY,test,test,33967,182117,105858,58.13,17.99,18.25,0.55,5.08,121.6,218.6,60.2,47.0,96.0,172.0,264.0,39.0,121.0,298.0,530.0,6.0,11.0,58.0,159.0
5,Delhi,DAILY,test,true test,30670,106534,73796,69.27,13.26,12.97,0.42,4.08,134.8,221.8,61.3,51.0,108.0,193.0,295.0,37.0,120.0,299.0,536.7,6.0,11.0,61.0,168.0
6,Delhi,MONTHLY,control,control,183552,453122,275975,60.91,15.78,16.70,0.87,5.75,133.1,325.8,75.6,56.0,107.0,185.0,279.0,68.0,198.0,431.0,736.0,6.0,11.0,60.0,183.0
7,Delhi,MONTHLY,test,test,14108,30010,17196,57.30,17.14,18.36,1.02,6.18,127.0,310.9,75.8,51.0,102.5,179.0,269.0,67.0,190.0,418.0,709.4,6.0,11.0,59.0,177.0
8,Delhi,MONTHLY,test,true test,12381,21090,13513,64.07,13.76,15.25,0.86,6.06,139.7,319.3,76.5,56.0,112.0,194.0,302.0,65.0,193.0,443.0,733.0,6.0,11.0,60.0,197.6
9,Delhi,RARE_NEED,control,control,180887,449883,272826,60.64,16.10,16.36,0.85,6.05,134.3,331.0,76.4,57.0,108.0,188.0,281.0,70.0,200.0,439.0,739.0,6.0,11.0,61.0,184.0


,city_name,taxi_regularity_segment,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.00,103.0,180.00,271.0,60.0,178.0,401.00,688.0,6.0,11.0,58.0,172.0
1,Delhi,BI_WEEKLY,test,test,30165,75527,39024,51.67,23.18,15.93,1.32,7.90,125.3,280.4,69.4,50.00,99.0,175.00,272.0,49.0,161.0,376.00,653.0,6.0,11.0,59.0,173.0
2,Delhi,BI_WEEKLY,test,true test,23109,39970,30982,77.51,2.18,18.67,0.07,1.57,163.2,323.0,72.7,78.00,140.0,226.00,323.8,69.0,197.0,435.00,724.8,6.0,10.0,58.0,181.0
3,Delhi,DAILY,control,control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.00,102.0,180.00,272.0,41.0,126.0,300.00,532.0,6.0,11.0,59.0,161.0
4,Delhi,DAILY,test,test,34378,195813,106494,54.39,22.51,16.08,0.71,6.32,123.6,209.2,59.4,47.00,98.0,175.00,269.0,35.0,112.5,282.00,510.0,6.0,11.0,59.0,158.0
5,Delhi,DAILY,test,true test,29746,92838,73160,78.80,3.04,16.78,0.06,1.32,155.8,240.5,62.6,63.00,133.0,219.00,324.0,45.0,141.0,326.00,571.0,6.0,11.0,60.0,169.0
6,Delhi,MONTHLY,control,control,183552,453122,275975,60.91,15.78,16.70,0.87,5.75,133.1,325.8,75.6,56.00,107.0,185.00,279.0,68.0,198.0,431.00,736.0,6.0,11.0,60.0,183.0
7,Delhi,MONTHLY,test,test,14899,33104,16701,50.45,23.28,16.23,1.47,8.56,129.9,304.3,75.2,52.00,104.0,182.00,277.0,61.0,182.5,411.75,700.7,6.0,11.0,60.0,178.0
8,Delhi,MONTHLY,test,true test,11353,17996,14008,77.84,1.88,18.63,0.01,1.64,170.9,329.5,77.3,91.75,150.5,235.25,320.9,74.0,203.0,454.00,753.8,6.0,10.0,59.0,194.1
9,Delhi,RARE_NEED,control,control,180887,449883,272826,60.64,16.10,16.36,0.85,6.05,134.3,331.0,76.4,57.00,108.0,188.00,281.0,70.0,200.0,439.00,739.0,6.0,11.0,61.0,184.0


,city_name,taxi_regularity_segment,sample_category,both_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.0,103.0,180.00,271.0,60.00,178.0,401.00,688.0,6.0,11.0,58.0,172.0
1,Delhi,BI_WEEKLY,test,test,32840,87770,48724,55.51,20.14,16.24,1.15,6.96,125.7,285.7,69.3,50.0,99.0,176.00,273.0,52.00,166.0,384.75,666.0,6.0,11.0,59.0,172.7
2,Delhi,BI_WEEKLY,test,true test,19257,27727,21282,76.76,2.53,18.92,0.06,1.74,163.4,326.7,74.3,83.0,141.0,220.00,318.0,69.00,198.0,435.00,724.5,6.0,10.0,59.0,188.0
3,Delhi,DAILY,control,control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.0,102.0,180.00,272.0,41.00,126.0,300.00,532.0,6.0,11.0,59.0,161.0
4,Delhi,DAILY,test,test,35437,232698,135876,58.39,19.26,16.26,0.60,5.48,124.3,215.6,59.7,47.0,98.0,177.00,271.0,37.00,118.0,292.00,524.0,6.0,11.0,58.0,159.0
5,Delhi,DAILY,test,true test,25585,55953,43778,78.24,3.69,16.49,0.07,1.51,152.6,235.9,63.6,62.0,129.0,217.00,319.0,43.00,136.5,319.00,569.0,6.0,11.0,61.0,174.0
6,Delhi,MONTHLY,control,control,183552,453122,275975,60.91,15.78,16.70,0.87,5.75,133.1,325.8,75.6,56.0,107.0,185.00,279.0,68.00,198.0,431.00,736.0,6.0,11.0,60.0,183.0
7,Delhi,MONTHLY,test,test,16377,38258,20751,54.24,20.33,16.56,1.27,7.60,130.1,307.7,75.2,52.0,104.0,182.50,277.0,64.00,187.0,416.50,707.6,6.0,11.0,59.0,178.0
8,Delhi,MONTHLY,test,true test,9472,12842,9958,77.54,2.08,18.62,0.01,1.75,176.0,330.6,78.2,95.0,153.0,241.50,332.0,70.00,199.0,454.00,757.0,6.0,10.0,59.0,201.0
9,Delhi,RARE_NEED,control,control,180887,449883,272826,60.64,16.10,16.36,0.85,6.05,134.3,331.0,76.4,57.0,108.0,188.00,281.0,70.00,200.0,439.00,739.0,6.0,11.0,61.0,184.0


,city_name,taxi_regularity_segment,sample_category,updated_segment,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,BI_WEEKLY,control,1. control,354006,1026703,627985,61.17,16.08,16.62,0.85,5.28,128.9,298.4,70.3,53.00,103.0,180.00,271.0,60.00,178.0,401.00,688.0,6.0,11.0,58.0,172.0
1,Delhi,BI_WEEKLY,test,2. not viewed,16933,31036,18122,58.39,19.38,13.57,1.17,7.49,133.0,282.6,68.6,51.00,103.0,186.00,294.0,50.00,167.0,387.00,667.0,6.0,11.0,59.0,175.0
2,Delhi,BI_WEEKLY,test,3. viewed,25267,56731,30601,53.94,20.55,17.69,1.13,6.68,121.9,286.9,69.7,49.00,97.0,171.00,262.0,52.00,166.0,383.00,666.0,6.0,11.0,58.0,171.0
3,Delhi,BI_WEEKLY,test,4. both_viewed,19254,27722,21278,76.75,2.53,18.92,0.06,1.74,163.4,326.7,74.2,83.00,141.0,220.00,318.0,69.00,198.0,435.00,724.6,6.0,10.0,59.0,187.0
4,Delhi,BI_WEEKLY,test,5. clicked,7,8,5,62.50,12.50,25.00,0.00,0.00,118.0,403.5,301.0,118.00,118.0,118.00,118.0,275.25,403.5,531.75,608.7,83.0,161.0,229.5,667.8
5,Delhi,DAILY,control,1. control,334267,2625158,1644553,62.65,16.30,15.93,0.55,4.57,127.7,222.6,60.6,52.00,102.0,180.00,272.0,41.00,126.0,300.00,532.0,6.0,11.0,59.0,161.0
6,Delhi,DAILY,test,2. not viewed,27552,87452,59392,67.91,14.65,12.51,0.49,4.44,133.7,224.6,59.6,50.00,106.0,192.00,294.0,38.00,123.0,306.00,539.0,6.0,11.0,60.0,161.0
7,Delhi,DAILY,test,3. viewed,31605,145232,76476,52.66,22.04,18.52,0.67,6.11,120.5,211.9,59.9,46.00,95.0,171.00,262.0,37.00,116.0,287.00,516.0,6.0,11.0,58.0,158.0
8,Delhi,DAILY,test,4. both_viewed,25582,55938,43766,78.24,3.69,16.48,0.07,1.51,152.6,235.8,63.6,62.00,129.0,217.00,319.0,43.00,136.0,319.00,569.0,6.0,11.0,61.0,174.0
9,Delhi,DAILY,test,5. clicked,26,29,20,68.97,3.45,17.24,0.00,10.34,104.0,585.0,169.5,104.00,104.0,104.00,104.0,186.00,548.0,635.00,1171.4,9.0,27.0,181.0,328.2


### Frequency

In [160]:
df_frequ = get_funnel(df_merge, ['city_name', 'dispatch_viewed_cux_tag', 'fq_bin'])

Execution time: 29.200 seconds


In [161]:
df_frequ 

,city_name,dispatch_viewed_cux_tag,fq_bin,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,NA,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.00,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,1-3,56841,130053,78117,60.07,15.49,18.05,0.70,5.69,123.5,288.5,68.4,48.0,98.0,175.0,267.0,52.00,161.0,378.0,674.0,6.0,10.0,56.0,166.0
2,Delhi,test,3-4,23169,83214,48917,58.78,16.50,18.37,0.77,5.57,121.7,258.2,63.7,46.0,96.0,171.0,266.0,46.00,145.0,341.0,605.0,6.0,11.0,58.0,162.0
3,Delhi,test,5-7,14867,79666,45417,57.01,18.68,18.13,0.70,5.47,125.2,238.5,63.3,50.0,100.0,176.0,270.0,44.00,135.0,324.0,570.0,6.0,11.0,60.0,162.0
4,Delhi,test,8+,8081,72979,38777,53.13,21.59,18.89,1.10,5.28,117.3,220.7,59.8,46.0,92.0,162.0,256.0,38.00,123.0,305.0,535.0,6.0,11.0,57.0,157.0
5,Delhi,test,NA,58706,108369,61821,57.05,15.86,19.56,0.84,6.69,121.5,308.5,77.9,48.0,98.0,172.0,259.0,56.00,177.0,409.0,719.0,6.0,11.0,59.0,185.0
6,Delhi,true test,1-3,90153,108884,76242,70.02,8.63,15.66,0.47,5.22,146.9,341.5,80.1,56.0,118.0,210.0,322.0,66.00,206.0,464.0,781.0,6.0,10.0,59.0,200.0
7,Delhi,true test,3-4,27061,70018,43033,61.46,14.81,16.08,0.83,6.82,137.6,268.7,67.1,52.0,111.0,197.0,300.0,50.00,156.0,363.0,632.0,6.0,11.0,60.0,179.0
8,Delhi,true test,5-7,15723,66775,42728,63.99,15.91,13.96,0.70,5.44,135.5,245.1,64.4,53.0,109.0,190.0,296.0,43.00,143.0,336.0,583.0,6.0,11.0,62.0,174.0
9,Delhi,true test,8+,8221,60460,37056,61.29,19.47,13.45,1.00,4.80,125.0,218.3,60.0,49.0,98.0,176.0,272.0,32.75,114.0,295.0,535.0,6.0,11.0,60.0,164.0


In [164]:
df_frequ_v1 = get_funnel(df_merge, ['city_name', 'sample_category', 'fq_bin'])

Execution time: 30.423 seconds


In [165]:
df_frequ_v1

,city_name,sample_category,fq_bin,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,NA,1933706,7054631,4301345,60.97,15.98,16.86,0.79,5.40,129.1,278.3,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.00,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,1-3,104028,238937,154359,64.60,12.36,16.96,0.60,5.48,131.0,310.8,74.0,50.0,104.0,187.0,284.0,58.0,177.0,412.00,721.0,6.0,10.0,57.0,180.0
2,Delhi,test,3-4,27535,153232,91950,60.01,15.73,17.33,0.80,6.14,128.5,262.7,65.3,49.0,102.0,182.0,281.0,48.0,149.0,350.25,618.0,6.0,11.0,59.0,170.0
3,Delhi,test,5-7,15774,146441,88145,60.19,17.42,16.23,0.70,5.46,129.5,241.1,63.8,51.0,103.0,181.0,280.0,43.0,138.0,328.00,575.0,6.0,11.0,61.0,168.0
4,Delhi,test,8+,8231,133439,75833,56.83,20.63,16.43,1.05,5.06,120.6,219.8,59.9,47.0,94.0,168.0,262.0,36.0,120.0,301.00,535.0,6.0,11.0,58.0,160.0
5,Delhi,test,NA,58706,108369,61821,57.05,15.86,19.56,0.84,6.69,121.5,308.5,77.9,48.0,98.0,172.0,259.0,56.0,177.0,409.00,719.0,6.0,11.0,59.0,185.0


In [166]:
df_frequ_v2 = get_funnel(df_merge[df_merge['hex_8_route'].isin(pickup_loc_hex8_route_list)], ['city_name', 'sample_category', 'fq_bin'])

Execution time: 23.043 seconds


In [167]:
df_frequ_v2

,city_name,sample_category,fq_bin,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,NA,1225689,3838586,2392905,62.34,16.68,15.70,0.48,4.81,126.6,253.7,63.8,52.0,101.0,177.0,267.0,48.0,145.0,340.0,594.0,6.0,11.0,58.0,161.0
1,Delhi,test,1-3,98255,183646,120985,65.88,11.98,16.23,0.50,5.41,131.8,308.3,73.8,51.0,104.0,188.0,286.0,58.0,177.5,412.0,715.0,6.0,10.0,57.0,180.0
2,Delhi,test,3-4,27434,126201,76011,60.23,15.96,16.72,0.73,6.36,129.7,256.7,64.8,49.0,102.0,184.0,284.0,47.0,146.0,345.0,609.0,6.0,11.0,59.0,170.0
3,Delhi,test,5-7,15758,126281,76856,60.86,17.59,15.49,0.62,5.44,130.8,236.4,63.5,52.0,105.0,183.0,284.0,43.0,135.0,324.0,562.0,6.0,11.0,62.0,169.0
4,Delhi,test,8+,8228,118526,67780,57.19,20.98,15.76,0.99,5.08,121.6,216.4,59.5,48.0,95.0,169.0,264.0,36.0,118.0,296.0,529.0,6.0,11.0,59.0,161.0
5,Delhi,test,NA,31148,52986,31583,59.61,16.36,18.14,0.39,5.51,116.3,284.6,68.3,46.0,94.0,163.0,248.2,53.0,164.0,382.0,657.0,6.0,11.0,56.0,162.0


In [148]:
df_merge.columns

Index(['experiment_name', 'execution_name', 'sample_category',
       'taxi_ltr_segment', 'taxi_retention_segments',
       'taxi_regularity_segment', 'taxi_income_segment', 'taxi_intent_segment',
       'yyyymmdd', 'customer_id', 'city_name', 'service_name', 'order_id',
       'modified_order_status', 'customer_cancelled_epoch',
       'order_requested_epoch', 'accepted_epoch', 'accept_to_cancelled',
       'cobra_ttc', 'ocara_ttc', 'tta', 'epoch', 'quarter_hour', 'hhmmss',
       'distance_final_distance', 'accept_to_pickup_distance',
       'pickup_location_hex_12', 'drop_location_hex_12',
       'pickup_location_hex_8', 'drop_location_hex_8', 'channel_host',
       'time_bucket', 'city', 'dispatch_viewed_cux', 'dispatch_clicked_cux',
       'otw_viewed_cux', 'otw_clicked_cux', 'both_viewed_cux', 'frequency',
       'gross_orders', 'accepted_orders', 'aor', 'mp_tag', 'fm_bin', 'lm_bin',
       'dispatch_viewed_cux_tag', 'dispatch_clicked_cux_tag',
       'otw_viewed_cux_tag', 'otw_c

### Route level

In [26]:
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,channel_host,time_bucket,city,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,both_viewed_cux,frequency,gross_orders,accepted_orders,aor,mp_tag,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_viewed_cux_tag,hex_8_route,clicked,both_view,single_view,updated_segment
0,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1747882233633,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,...,android,morning_peak,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2783.0,2394.0,86.02,1. Good,0-500m,6-9km,control,control,control,control,control,883da1118dfffff - 883da1034dfffff,None,None,None,1. control


In [88]:
list_hex_8_route =  df_merge\
                        .groupby(['hex_8_route'])\
                        .agg(orders=('order_id', 'nunique'))\
                        .reset_index()\
                        .sort_values(by='orders', ascending=False)\
                        .head(10)['hex_8_route'].tolist()

In [89]:
list_hex_8_route

['883da11a83fffff - 883da11abdfffff',
 '883da11abdfffff - 883da11a83fffff',
 '883da102c5fffff - 883da102c7fffff',
 '883da102c7fffff - 883da102c5fffff',
 '883da102e5fffff - 883da102e9fffff',
 '883da102e9fffff - 883da102e5fffff',
 '883da11887fffff - 883da1188bfffff',
 '883da1188bfffff - 883da11883fffff',
 '883da102e1fffff - 883da102e9fffff',
 '883da111a3fffff - 883da111a1fffff']

In [90]:
get_funnel(df_merge[df_merge['hex_8_route'].isin(list_hex_8_route)], ['city_name', 'sample_category', 'dispatch_viewed_cux_tag'])

Execution time: 0.163 seconds


,city_name,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,9831,24281,14987,61.72,21.12,13.17,0.14,3.85,125.9,181.5,56.3,53.0,99.0,172.00,266.0,34.00,97.0,251.75,453.0,6.0,11.0,59.0,149.7
1,Delhi,test,test,845,1640,1019,62.13,20.85,13.29,0.06,3.66,118.1,175.7,46.8,47.0,92.0,163.75,245.9,37.25,94.5,222.75,464.5,6.0,10.0,48.5,117.4
2,Delhi,test,true test,616,1076,677,62.92,22.40,11.43,0.09,3.16,132.8,200.5,56.2,52.0,93.0,200.00,296.0,36.50,94.0,283.50,575.2,6.0,12.0,58.5,127.6


In [143]:
pickup_loc_hex8_list = df_merge[df_merge['dispatch_viewed_cux_tag'].isin(['true test'])]['pickup_location_hex_8'].tolist()

In [144]:
df_pickup_loc = get_funnel(df_merge[df_merge['pickup_location_hex_8'].isin(pickup_loc_hex8_list)], ['city_name','sample_category', 'dispatch_viewed_cux_tag'])

Execution time: 40.536 seconds


In [145]:
df_pickup_loc

,city_name,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1929745,7033834,4296160,61.08,15.97,16.87,0.74,5.34,129.0,278.0,68.7,53.0,103.0,181.0,273.0,52.0,159.0,370.0,650.0,6.0,11.0,58.0,171.0
1,Delhi,test,test,161401,473231,272806,57.65,17.22,18.60,0.77,5.75,121.8,269.1,67.6,48.0,97.0,172.0,263.0,48.0,151.0,358.0,632.0,6.0,11.0,58.0,167.0
2,Delhi,test,true test,141158,306137,199059,65.02,13.77,14.95,0.71,5.55,135.6,282.0,70.2,52.0,108.0,192.0,296.0,50.0,160.0,381.0,664.0,6.0,11.0,60.0,181.0


In [146]:
df_pickup_loc_mp = get_funnel(df_merge[df_merge['pickup_location_hex_8'].isin(pickup_loc_hex8_list)], ['city_name','mp_tag', 'sample_category', 'dispatch_viewed_cux_tag'])

Execution time: 40.061 seconds


In [147]:
df_pickup_loc_mp

,city_name,mp_tag,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,1. Good,control,control,1248863,3156131,2131067,67.52,11.16,17.45,0.43,3.44,114.1,279.0,61.1,46.00,89.0,157.00,245.0,51.00,159.0,371.00,649.0,6.0,9.0,46.0,139.0
1,Delhi,1. Good,test,test,102104,222098,141931,63.90,12.65,19.34,0.41,3.69,107.9,266.3,60.3,42.00,85.0,149.00,235.0,46.00,149.0,358.00,628.0,6.0,9.0,46.0,136.0
2,Delhi,1. Good,test,true test,75575,126990,91766,72.26,8.93,15.24,0.39,3.17,119.2,281.9,60.5,44.00,92.0,165.00,270.0,49.00,162.0,380.00,658.3,5.0,9.0,42.0,140.0
3,Delhi,2. Balance,control,control,991670,2461709,1479237,60.09,16.84,17.15,0.65,5.27,126.2,278.6,71.8,52.00,101.0,176.00,266.0,52.00,159.0,372.00,656.0,6.0,12.0,64.0,179.0
4,Delhi,2. Balance,test,test,77897,165131,93247,56.47,18.25,18.92,0.63,5.73,121.6,271.7,71.3,48.00,98.0,172.00,261.0,49.00,153.0,360.00,640.0,6.0,12.0,64.0,178.0
5,Delhi,2. Balance,test,true test,61294,106990,69395,64.86,14.14,14.92,0.60,5.47,131.5,286.0,72.7,50.00,105.0,186.00,287.0,50.00,162.0,387.00,682.8,6.0,12.0,66.0,189.0
6,Delhi,3. Poor,control,control,504710,1390934,678482,48.78,25.26,15.07,1.38,9.51,147.1,271.5,84.3,63.00,121.0,208.00,302.0,54.00,158.0,362.00,635.0,6.0,16.0,95.0,230.0
7,Delhi,3. Poor,test,test,35662,83930,37111,44.22,27.17,16.10,1.68,10.83,138.7,269.6,84.3,56.00,113.0,198.00,293.0,52.00,154.0,348.00,618.0,6.0,16.0,95.0,229.0
8,Delhi,3. Poor,test,true test,36033,70407,37412,53.14,21.85,14.48,1.06,9.47,151.5,273.9,87.7,63.00,125.0,215.00,320.0,51.00,154.0,367.00,640.5,6.0,16.0,100.0,238.0
9,Delhi,NaN,control,control,13604,25060,7374,29.43,20.66,16.80,12.14,20.97,152.5,398.6,109.4,57.00,118.0,221.00,331.0,80.00,235.0,542.00,926.2,6.0,11.0,77.0,293.0


In [152]:
pickup_loc_hex8_route_list = df_merge[df_merge['dispatch_viewed_cux_tag'].isin(['true test'])]['hex_8_route'].tolist()

In [153]:
df_hex8_route = get_funnel(df_merge[df_merge['hex_8_route'].isin(pickup_loc_hex8_route_list)], ['city_name','sample_category', 'dispatch_viewed_cux_tag'])

Execution time: 17.586 seconds


In [154]:
df_hex8_route

,city_name,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,1225689,3838586,2392905,62.34,16.68,15.70,0.48,4.81,126.6,253.7,63.8,52.0,101.0,177.0,267.0,48.0,145.0,340.0,594.0,6.0,11.0,58.0,161.0
1,Delhi,test,test,109720,301503,174156,57.76,18.49,17.58,0.61,5.57,120.8,247.2,63.0,48.0,95.0,170.0,261.0,45.0,139.0,332.0,586.0,6.0,11.0,58.0,159.0
2,Delhi,test,true test,141158,306137,199059,65.02,13.77,14.95,0.71,5.55,135.6,282.0,70.2,52.0,108.0,192.0,296.0,50.0,160.0,381.0,664.0,6.0,11.0,60.0,181.0


### PrePost

In [136]:
df_preperiod['unique_id_pickupid'] = df_preperiod['customer_id'] + ' ' + df_preperiod['pickup_location_hex_8']
df_preperiod.head(1)

,sample_category,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,pickup_location_hex_12,drop_location_hex_12,pickup_location_hex_8,drop_location_hex_8,channel_host,time_bucket,date_type,unique_id_pickupid
0,test,20250515,638b5719f470701c3f8ad582,Delhi,Bike Lite,6825f7a7ea13344195ff271b,OCARA,1.747319e+12,1.747319e+12,1.747319e+12,0.0,NaN,278.0,199.0,1747318695281,1945,194815,9.8,0.468,8c3da115056dbff,8c3da10287737ff,883da11501fffff,883da10287fffff,android,evening_peak,pre-period,638b5719f470701c3f8ad582 883da11501fffff


In [137]:
df_merge['unique_id_pickupid'] = df_merge['customer_id'] + ' ' + df_merge['pickup_location_hex_8']
df_merge['date_type'] = 'post-period'
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,both_viewed_cux,frequency,gross_orders,accepted_orders,aor,mp_tag,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_viewed_cux_tag,hex_8_route,clicked,both_view,single_view,updated_segment,orders_per_customers,order_bin,date_type,unique_id_pickupid
0,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1747882233633,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,...,NaN,NaN,NaN,NaN,NaN,2783.0,2394.0,86.02,1. Good,0-500m,6-9km,control,control,control,control,control,883da1118dfffff - 883da1034dfffff,None,None,None,1. control,3,NA,post-period,5dffc63387a252727580ccac 883da1118dfffff


In [138]:
# Get the intersection of the two columns
intersection_ids = set(df_merge['unique_id_pickupid']).intersection(df_preperiod['unique_id_pickupid'])

# Create a new DataFrame with the intersection ids
df_intersection = pd.DataFrame({'unique_id_pickupid': list(intersection_ids)})
prepost_intersection = df_intersection['unique_id_pickupid'].tolist()

In [139]:
df_pre = get_funnel(df_preperiod[df_preperiod['unique_id_pickupid'].isin(prepost_intersection)], ['date_type', 'city_name'])
df_post = get_funnel(df_merge[df_merge['unique_id_pickupid'].isin(prepost_intersection)], ['date_type', 'city_name'])
df_prepost = pd.concat([df_pre, df_post], axis=0)

Execution time: 1.022 seconds
Execution time: 1.192 seconds


In [140]:
df_prepost

,date_type,city_name,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,pre-period,Delhi,79760,351194,226618,64.53,15.66,15.32,0.48,4.01,122.9,229.9,61.6,48.0,98.0,174.0,264.0,40.0,128.0,313.0,554.0,6.0,12.0,59.0,161.0
0,post-period,Delhi,79760,356181,224194,62.94,16.34,15.31,0.53,4.88,127.7,233.1,62.6,49.0,101.0,181.0,278.0,41.0,131.0,319.0,563.0,6.0,11.0,60.0,167.0


In [134]:
df_prepost

,date_type,city_name,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,pre-period,Delhi,58441,231218,150893,65.26,15.53,14.79,0.40,4.02,125.1,224.0,61.1,49.0,100.0,177.0,268.0,39.0,124.0,304.0,540.0,6.0,12.0,60.0,162.0
0,post-period,Delhi,58441,232427,148381,63.84,16.22,14.69,0.45,4.79,130.2,222.9,61.8,51.0,103.0,185.0,284.0,39.0,124.0,305.0,541.0,6.0,11.0,61.0,168.0


### Time distribution

In [67]:
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,channel_host,time_bucket,city,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,both_viewed_cux,frequency,gross_orders,accepted_orders,aor,mp_tag,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_viewed_cux_tag,hex_8_route,clicked,both_view,single_view,updated_segment
0,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1747882233633,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,...,android,morning_peak,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2783.0,2394.0,86.02,1. Good,0-500m,6-9km,control,control,control,control,control,883da1118dfffff - 883da1034dfffff,None,None,None,1. control


In [61]:
df_merge.updated_segment.unique()

array(['1. control', '3. viewed', '4. both_viewed', '2. not viewed',
       '5. clicked'], dtype=object)

In [171]:
df_distr = df_merge[(df_merge['sample_category'].isin(['control'])) | (df_merge['dispatch_viewed_cux_tag'].isin(['true test']))]\
[['sample_category', 'city_name', 'customer_id', 'order_id', 'modified_order_status', 'cobra_ttc', 'tta', 'updated_segment']].reset_index(drop=True)

In [172]:
df_distr['end_time'] = np.where(df_distr['tta'].isna(), df_distr['cobra_ttc'], df_distr['tta'])
df_distr['end_time'] = df_distr['end_time'].fillna(300.0)
df_distr

,sample_category,city_name,customer_id,order_id,modified_order_status,cobra_ttc,tta,updated_segment,end_time
0,control,Delhi,5dffc63387a252727580ccac,682e90f919e59a597fb36199,dropped,NaN,7.0,1. control,7.0
1,control,Delhi,5dffc63387a252727580ccac,682eeaf88f45c100810348c9,dropped,NaN,6.0,1. control,6.0
2,control,Delhi,675460206841a4df6b4aa9bd,682edca59ddad10070bc80fa,dropped,NaN,24.0,1. control,24.0
3,control,Delhi,604664d47699f6661479a5d1,682f5b46d657147021309148,OCARA,NaN,7.0,1. control,7.0
4,control,Delhi,604664d47699f6661479a5d1,682f59b6d657147021308cc6,OCARA,NaN,4.0,1. control,4.0
...,...,...,...,...,...,...,...,...,...
7360763,control,Delhi,637e611927c3ff45b4851197,682d51a6f82a504357ea8ae0,dropped,NaN,499.0,1. control,499.0
7360764,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4965acc02e2acb5ceccf,COBRA,138.0,NaN,1. control,138.0
7360765,control,Delhi,5d8db5d7a1f0a71c604ab047,682d492609b90e311db9a1a8,expired,NaN,NaN,1. control,300.0
7360766,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4a10f9fb0423027fa162,COBRA,337.0,NaN,1. control,337.0


In [173]:
check_bins = [0, 10, 30, 60, 90, 120, 180, 240, 300, float('inf')]
check_labels = ['0-10', '10-30', '30-60', '60-90', '90-120', '120-180', '180-240', '240-300', '300+']

# Create a new column with the binned data
df_distr['end_time_bin'] = pd.cut(df_distr['end_time'], bins=check_bins, labels=check_labels, right=False)

df_distr

,sample_category,city_name,customer_id,order_id,modified_order_status,cobra_ttc,tta,updated_segment,end_time,end_time_bin
0,control,Delhi,5dffc63387a252727580ccac,682e90f919e59a597fb36199,dropped,NaN,7.0,1. control,7.0,0-10
1,control,Delhi,5dffc63387a252727580ccac,682eeaf88f45c100810348c9,dropped,NaN,6.0,1. control,6.0,0-10
2,control,Delhi,675460206841a4df6b4aa9bd,682edca59ddad10070bc80fa,dropped,NaN,24.0,1. control,24.0,10-30
3,control,Delhi,604664d47699f6661479a5d1,682f5b46d657147021309148,OCARA,NaN,7.0,1. control,7.0,0-10
4,control,Delhi,604664d47699f6661479a5d1,682f59b6d657147021308cc6,OCARA,NaN,4.0,1. control,4.0,0-10
...,...,...,...,...,...,...,...,...,...,...
7360763,control,Delhi,637e611927c3ff45b4851197,682d51a6f82a504357ea8ae0,dropped,NaN,499.0,1. control,499.0,300+
7360764,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4965acc02e2acb5ceccf,COBRA,138.0,NaN,1. control,138.0,120-180
7360765,control,Delhi,5d8db5d7a1f0a71c604ab047,682d492609b90e311db9a1a8,expired,NaN,NaN,1. control,300.0,300+
7360766,control,Delhi,5d8db5d7a1f0a71c604ab047,682d4a10f9fb0423027fa162,COBRA,337.0,NaN,1. control,337.0,300+


In [174]:
df_testing = df_distr.groupby(['sample_category', 'end_time_bin', 'modified_order_status']).agg(orders = ('order_id', 'nunique')).reset_index()
df_testing.pivot(index=['sample_category', 'modified_order_status'], columns=['end_time_bin'], values=['orders'])

orders                          \
end_time_bin                              0-10   10-30   30-60   60-90   
sample_category modified_order_status                                    
control         COBRA                    33112  107050  184526  169854   
                COBRM                        0       0       0       0   
                OCARA                   449958  242877   95393   75881   
                aborted                   2003    1281     509     396   
                dropped                2093459  930406  339229  248986   
                expired                   6905    6047    3165    2604   
                requested                    0       0       0       0   
test            COBRA                     1680    4079    6401    5679   
                COBRM                        0       0       0       0   
                OCARA                    18227    8702    3346    2614   
                aborted                     55      46      13      18   
                dropped                  97425   41389   14796   11359   
                expired                    391     303     149     129   
                requested                    0       0       0       0   

                                                                               
end_time_bin                           90-120 120-180 180-240 240-300    300+  
sample_category modified_order_status                                          
control         COBRA                  144894  202067  124112   79581   82075  
                COBRM                       0       0       0       0   55498  
                OCARA                   59265   70212   45982   32735  117199  
                aborted                   287     394     278     173    1163  
                dropped                186526  188155  106715   65566  142303  
                expired                  2286    3297    2569    2016  345612  
                requested                   0       0       0       0      30  
test            COBRA                    5092    7541    4598    3002    4091  
                COBRM                       0       0       0       0    2163  
                OCARA                    2213    2654    1769    1309    4931  
                aborted                     6      14       7       4      30  
                dropped                  8879    9000    5445    3439    7327  
                expired                   131     168     114      95   15314  
                requested                   0       0       0       0       0

### Patience

In [178]:
df_merge['end_time'] = np.where(df_merge['tta'].isna(), df_merge['cobra_ttc'], df_merge['tta'])
df_merge['end_time'] = df_merge['end_time'].fillna(300.0)
df_merge[['end_time']].head(1)

,end_time
0,7.0


In [208]:
thresholds = [30, 60, 90, 120, 180, 240]

for threshold in thresholds:
    wait_col = f'patience_tag_{threshold}_wait'
    cancel_col = f'patience_tag_{threshold}_cancel'

    df_merge[wait_col] = np.where(
        ((df_merge['cobra_ttc'] > threshold) | (df_merge['cobra_ttc'].isna())) 
        &
        ((df_merge['tta'] > threshold) | df_merge['tta'].isna()),
        df_merge['order_id'],
        None
    )

    df_merge[cancel_col] = np.where(
        (df_merge['cobra_ttc'] <= threshold),
        df_merge['order_id'],
        None
    )
    df_merge[f'patience_tag_{threshold}_denom'] = df_merge[wait_col].fillna(df_merge[cancel_col])

In [209]:
df_merge[['order_id', 'cobra_ttc', 'tta', 'ocara_ttc']].head(10)

,order_id,cobra_ttc,tta,ocara_ttc
0,682e90f919e59a597fb36199,NaN,7.0,NaN
1,682eeaf88f45c100810348c9,NaN,6.0,NaN
2,682edca59ddad10070bc80fa,NaN,24.0,NaN
3,682f5b46d657147021309148,NaN,7.0,217.0
4,682f59b6d657147021308cc6,NaN,4.0,364.0
5,682f1c75206b5f3963b190dd,NaN,5.0,NaN
6,682ea880483ddc524942a536,NaN,131.0,755.0
7,682ea7b0a24a2d58a0c9f13c,27.0,NaN,NaN
8,682eabfbfed17e0062be1a28,NaN,84.0,NaN
9,682ea7cfe3d8853c7da55da0,26.0,NaN,NaN


In [210]:
df_merge.columns

Index(['experiment_name', 'execution_name', 'sample_category',
       'taxi_ltr_segment', 'taxi_retention_segments',
       'taxi_regularity_segment', 'taxi_income_segment', 'taxi_intent_segment',
       'yyyymmdd', 'customer_id', 'city_name', 'service_name', 'order_id',
       'modified_order_status', 'customer_cancelled_epoch',
       'order_requested_epoch', 'accepted_epoch', 'accept_to_cancelled',
       'cobra_ttc', 'ocara_ttc', 'tta', 'epoch', 'quarter_hour', 'hhmmss',
       'distance_final_distance', 'accept_to_pickup_distance',
       'pickup_location_hex_12', 'drop_location_hex_12',
       'pickup_location_hex_8', 'drop_location_hex_8', 'channel_host',
       'time_bucket', 'city', 'dispatch_viewed_cux', 'dispatch_clicked_cux',
       'otw_viewed_cux', 'otw_clicked_cux', 'both_viewed_cux', 'frequency',
       'gross_orders', 'accepted_orders', 'aor', 'mp_tag', 'fm_bin', 'lm_bin',
       'dispatch_viewed_cux_tag', 'dispatch_clicked_cux_tag',
       'otw_viewed_cux_tag', 'otw_c

In [211]:
df_patience_bins = df_merge[df_merge['hex_8_route'].isin(pickup_loc_hex8_route_list)]\
                        .groupby(['city_name', 'sample_category', 'dispatch_viewed_cux_tag'])\
                        .agg(patience_tag_30_wait = ('patience_tag_30_wait', 'nunique'),
                             patience_tag_30_denom = ('patience_tag_30_denom', 'nunique'),
                             patience_tag_60_wait = ('patience_tag_60_wait', 'nunique'),
                             patience_tag_60_denom = ('patience_tag_60_denom', 'nunique'),
                             patience_tag_90_wait = ('patience_tag_90_wait', 'nunique'),
                             patience_tag_90_denom = ('patience_tag_90_denom', 'nunique'),
                             patience_tag_120_wait = ('patience_tag_120_wait', 'nunique'),
                             patience_tag_120_denom = ('patience_tag_120_denom', 'nunique'),
                             patience_tag_180_wait = ('patience_tag_180_wait', 'nunique'),
                             patience_tag_180_denom = ('patience_tag_180_denom', 'nunique'),
                             patience_tag_240_wait = ('patience_tag_240_wait', 'nunique'),
                             patience_tag_240_denom = ('patience_tag_240_denom', 'nunique'),
                            )\
                        .reset_index()

In [212]:
df_patience_bins['patience_tag_30'] = (df_patience_bins['patience_tag_30_wait']*100.00/df_patience_bins['patience_tag_30_denom']).round(2)
df_patience_bins['patience_tag_60'] = (df_patience_bins['patience_tag_60_wait']*100.00/df_patience_bins['patience_tag_60_denom']).round(2)
df_patience_bins['patience_tag_90'] = (df_patience_bins['patience_tag_90_wait']*100.00/df_patience_bins['patience_tag_90_denom']).round(2)
df_patience_bins['patience_tag_120'] = (df_patience_bins['patience_tag_120_wait']*100.00/df_patience_bins['patience_tag_120_denom']).round(2)
df_patience_bins['patience_tag_180'] = (df_patience_bins['patience_tag_180_wait']*100.00/df_patience_bins['patience_tag_180_denom']).round(2)
df_patience_bins['patience_tag_240'] = (df_patience_bins['patience_tag_240_wait']*100.00/df_patience_bins['patience_tag_240_denom']).round(2)

In [207]:
df_patience_bins[['city_name', 'sample_category', 'dispatch_viewed_cux_tag', 
                  'patience_tag_30', 'patience_tag_60', 'patience_tag_90', 'patience_tag_120', 'patience_tag_180', 'patience_tag_240' ]]

,city_name,sample_category,dispatch_viewed_cux_tag,patience_tag_30,patience_tag_60,patience_tag_90,patience_tag_120,patience_tag_180,patience_tag_240
0,Delhi,control,control,95.37,87.75,79.01,69.84,55.40,44.78
1,Delhi,test,test,94.12,85.77,76.63,67.27,53.25,43.39
2,Delhi,test,true test,95.71,89.74,83.14,75.84,63.59,54.19


In [180]:
df_merge.shape

(7835049, 61)

### Sample Biasness Validate

#### Control VS Test 90:10 

In [30]:
# taxi_regularity_segment

sample_category = ['test', 'control']

for city_name, execution_name in  city.items():
    for group_tc in sample_category:
        print(city_name, group_tc)
        print(df_merge.query(f"execution_name == '{execution_name}' & sample_category == '{group_tc}'").taxi_regularity_segment.value_counts(normalize=True))

Bangalore test
Series([], Name: proportion, dtype: float64)
Bangalore control
Series([], Name: proportion, dtype: float64)
Delhi test
taxi_regularity_segment
DAILY        0.390978
WEEKLY       0.234698
BI_WEEKLY    0.156441
UNKNOWN      0.081339
MONTHLY      0.069215
RARE_NEED    0.067329
Name: proportion, dtype: float64
Delhi control
taxi_regularity_segment
DAILY        0.393334
WEEKLY       0.236159
BI_WEEKLY    0.153833
UNKNOWN      0.081375
MONTHLY      0.067892
RARE_NEED    0.067407
Name: proportion, dtype: float64
Hyderabad test
Series([], Name: proportion, dtype: float64)
Hyderabad control
Series([], Name: proportion, dtype: float64)


In [31]:
# taxi_retention_segments

sample_category = ['test', 'control']

for city_name, execution_name in  city.items():
    for group_tc in sample_category:
        print(city_name, group_tc)
        print(df_merge.query(f"execution_name == '{execution_name}' & sample_category == '{group_tc}'").taxi_retention_segments.value_counts(normalize=True))

Bangalore test
Series([], Name: proportion, dtype: float64)
Bangalore control
Series([], Name: proportion, dtype: float64)
Delhi test
taxi_retention_segments
ELITE       0.431132
PLATINUM    0.199902
GOLD        0.120519
DORMANT     0.104657
SILVER      0.051062
INACTIVE    0.043080
HH          0.039026
PRIME       0.010622
Name: proportion, dtype: float64
Delhi control
taxi_retention_segments
ELITE       0.434657
PLATINUM    0.197867
GOLD        0.120248
DORMANT     0.104972
SILVER      0.049951
INACTIVE    0.042720
HH          0.039145
PRIME       0.010440
Name: proportion, dtype: float64
Hyderabad test
Series([], Name: proportion, dtype: float64)
Hyderabad control
Series([], Name: proportion, dtype: float64)


In [32]:
# taxi_retention_segments

sample_category = ['test', 'control']

for city_name, execution_name in  city.items():
    for group_tc in sample_category:
        print(city_name, group_tc)
        print(df_merge.query(f"execution_name == '{execution_name}' & sample_category == '{group_tc}'").service_name.value_counts(normalize=True))

Bangalore test
Series([], Name: proportion, dtype: float64)
Bangalore control
Series([], Name: proportion, dtype: float64)
Delhi test
service_name
Auto          0.264415
Link          0.210449
Bike Lite     0.208208
CabEconomy    0.193282
Bike Metro    0.078800
CabPremium    0.022183
Cab SUV       0.011546
Auto NCR      0.011117
Name: proportion, dtype: float64
Delhi control
service_name
Auto          0.262647
Link          0.212419
Bike Lite     0.210199
CabEconomy    0.191824
Bike Metro    0.078360
CabPremium    0.021620
Cab SUV       0.011570
Auto NCR      0.011361
Name: proportion, dtype: float64
Hyderabad test
Series([], Name: proportion, dtype: float64)
Hyderabad control
Series([], Name: proportion, dtype: float64)


#### ReSampling

In [103]:
df_merge.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,city,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,both_viewed_cux,frequency,gross_orders,accepted_orders,aor,mp_tag,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_viewed_cux_tag,hex_8_route,clicked,both_view,single_view,updated_segment,orders_per_customers,order_bin
0,loyalty_experiment2,loyalty_experiment2_delhi,control,PHH,ELITE,WEEKLY,UNKNOWN,MEDIUM,20250522,5dffc63387a252727580ccac,Delhi,Bike Metro,682e90f919e59a597fb36199,dropped,NaN,1747882233633,1.747882e+12,NaN,NaN,NaN,7.0,1747882233633,0815,082033,6.98,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2783.0,2394.0,86.02,1. Good,0-500m,6-9km,control,control,control,control,control,883da1118dfffff - 883da1034dfffff,None,None,None,1. control,3,NA


In [34]:
df_merge.updated_segment.unique()

array(['1. control', '3. viewed', '4. both_viewed', '2. not viewed',
       '5. clicked'], dtype=object)

In [107]:
group_cols = ['taxi_regularity_segment', 'taxi_retention_segments', 'modified_order_status', 'orders_per_customers']
viewed_test = df_merge[df_merge['updated_segment'].isin(['3. viewed'])]

# Calculate proportions of the viewed test group
dist = viewed_test.groupby(group_cols).size().div(len(viewed_test))

In [108]:
# Total number of samples to draw
total_samples = len(viewed_test)

# Normalize proportions to sum to 1 (in case of rounding error)
normalized_dist = dist / dist.sum()

# Compute exact number of samples per group that sum to total_samples
group_sample_sizes = (normalized_dist * total_samples).round().astype(int)

resampled_control = pd.DataFrame()

for group_values, n_samples in tqdm(group_sample_sizes.items()):
    if not isinstance(group_values, tuple):
        group_values = (group_values,)

    mask = pd.Series(True, index=control.index)
    for col, val in zip(group_cols, group_values):
        mask &= (control[col] == val)

    candidates = control[mask]
    sample = candidates.sample(n=min(n_samples, len(candidates)), replace=False)
    resampled_control = pd.concat([resampled_control, sample])

# Trim or pad resampled_control to match viewed_test size exactly
if len(resampled_control) > len(viewed_test):
    resampled_control = resampled_control.sample(n=len(viewed_test), random_state=42)
elif len(resampled_control) < len(viewed_test):
    deficit = len(viewed_test) - len(resampled_control)
    additional = control.sample(n=deficit, replace=False, random_state=42)
    resampled_control = pd.concat([resampled_control, additional])


0it [00:01, ?it/s]


KeyError: 'orders_per_customers'

In [46]:
# Total number of samples to draw
total_samples = len(viewed_test)

# Normalize proportions to sum to 1 (in case of rounding error)
normalized_dist = dist / dist.sum()

# Compute exact number of samples per group that sum to total_samples
group_sample_sizes = (normalized_dist * total_samples).round().astype(int)

resampled_control = pd.DataFrame()

for group_values, n_samples in tqdm(group_sample_sizes.items()):
    if not isinstance(group_values, tuple):
        group_values = (group_values,)

    mask = pd.Series(True, index=control.index)
    for col, val in zip(group_cols, group_values):
        mask &= (control[col] == val)

    candidates = control[mask]
    sample = candidates.sample(n=min(n_samples, len(candidates)), replace=False)
    resampled_control = pd.concat([resampled_control, sample])

# Trim or pad resampled_control to match viewed_test size exactly
if len(resampled_control) > len(viewed_test):
    resampled_control = resampled_control.sample(n=len(viewed_test), random_state=42)
elif len(resampled_control) < len(viewed_test):
    deficit = len(viewed_test) - len(resampled_control)
    additional = control.sample(n=deficit, replace=False, random_state=42)
    resampled_control = pd.concat([resampled_control, additional])


1299it [39:04,  1.80s/it]


In [81]:
# Total number of samples to draw
total_samples = len(viewed_test)

# Normalize proportions to sum to 1 (in case of rounding error)
normalized_dist = dist / dist.sum()

# Compute exact number of samples per group that sum to total_samples
group_sample_sizes = (normalized_dist * total_samples).round().astype(int)

resampled_control = pd.DataFrame()

for group_values, n_samples in tqdm(group_sample_sizes.items()):
    if not isinstance(group_values, tuple):
        group_values = (group_values,)

    mask = pd.Series(True, index=control.index)
    for col, val in zip(group_cols, group_values):
        mask &= (control[col] == val)

    candidates = control[mask]
    sample = candidates.sample(n=min(n_samples, len(candidates)), replace=False)
    resampled_control = pd.concat([resampled_control, sample])

# Trim or pad resampled_control to match viewed_test size exactly
if len(resampled_control) > len(viewed_test):
    resampled_control = resampled_control.sample(n=len(viewed_test), random_state=42)
elif len(resampled_control) < len(viewed_test):
    deficit = len(viewed_test) - len(resampled_control)
    additional = control.sample(n=deficit, replace=False, random_state=42)
    resampled_control = pd.concat([resampled_control, additional])


1331it [50:14,  2.26s/it]


In [35]:
# Step 1: Match customer_id distribution (same as before)
group_cols = ['taxi_regularity_segment', 'taxi_retention_segments', 'service_name', 'modified_order_status']
viewed_test = df_merge[~df_merge['updated_segment'].isin(['1. control', '2. not viewed'])]

# Unique customers per group in test
viewed_test_customers = viewed_test.groupby(group_cols)['customer_id'].unique().explode().reset_index()

# Count unique customers per group
customer_dist = viewed_test_customers.groupby(group_cols)['customer_id'].nunique()
normalized_dist = customer_dist / customer_dist.sum()
total_customers_needed = viewed_test['customer_id'].nunique()
group_sample_sizes = (normalized_dist * total_customers_needed).round().astype(int)

control = df_merge[df_merge['sample_category'] == 'control']
resampled_customers = pd.DataFrame()


In [36]:

# Step 2: Sample unique customers from control
for group_values, n_customers in tqdm(group_sample_sizes.items()):
    if not isinstance(group_values, tuple):
        group_values = (group_values,)
    
    mask = pd.Series(True, index=control.index)
    for col, val in zip(group_cols, group_values):
        mask &= (control[col] == val)
    
    candidates = control[mask]['customer_id'].drop_duplicates()
    sampled_customers = candidates.sample(n=min(n_customers, len(candidates)), replace=False)
    resampled_customers = pd.concat([resampled_customers, sampled_customers.to_frame()])

1331it [34:40,  1.56s/it]


In [37]:
# Step 3: Match order counts per customer
# Count orders per customer in viewed_test
test_order_counts = viewed_test.groupby('customer_id').size()

resampled_control = pd.DataFrame()
for cust_id, n_orders in tqdm(test_order_counts.items()):
    cust_orders = control[control['customer_id'] == cust_id]
    if len(cust_orders) == 0:
        continue
    sample = cust_orders.sample(n=min(n_orders, len(cust_orders)), replace=False)
    resampled_control = pd.concat([resampled_control, sample])

53742it [6:06:00,  2.45it/s]


KeyboardInterrupt: 

In [47]:
# Tag groups
viewed_test = viewed_test.copy()
viewed_test['group_type'] = 'test_viewed'

resampled_control = resampled_control.copy()
resampled_control['group_type'] = 'control_matched'

# Combine
comparison_df = pd.concat([viewed_test, resampled_control], ignore_index=True)

# ✅ Final checks
print("Unique customer_id in test:   ", viewed_test['customer_id'].nunique())
print("Unique customer_id in control:", resampled_control['customer_id'].nunique())
print("Total orders in test:         ", len(viewed_test))
print("Total orders in control:      ", len(resampled_control))

Unique customer_id in test:    143676
Unique customer_id in control: 316970
Total orders in test:          387501
Total orders in control:       387501


In [38]:
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

# Assuming your data is in df_merge
group_cols = ['taxi_regularity_segment', 'taxi_retention_segments', 'service_name', 'modified_order_status']

# 1️⃣ Filter test data: viewed_test
viewed_test = df_merge[~df_merge['updated_segment'].isin(['1. control', '2. not viewed'])].copy()
viewed_test['group_type'] = 'test_viewed'

# 2️⃣ Compute order count per customer in test
test_order_counts = viewed_test.groupby('customer_id').size()

# 3️⃣ Control pool: exclude test customers
test_customer_ids = set(viewed_test['customer_id'])
control = df_merge[
    (df_merge['sample_category'] == 'control') &
    (~df_merge['customer_id'].isin(test_customer_ids))
].copy()


In [39]:

# 4️⃣ Efficient sampling: match order count per customer
# Filter to only control customers present in test_order_counts
eligible_control = control[control['customer_id'].isin(test_order_counts.index)]

def sample_orders(group):
    cust_id = group.name
    n_orders_needed = test_order_counts.get(cust_id, 0)
    if n_orders_needed == 0:
        return pd.DataFrame()  # skip if not needed
    n_samples = min(n_orders_needed, len(group))
    return group.sample(n=n_samples, replace=False, random_state=42)

In [ ]:


# Efficient sampling using groupby + apply
resampled_control = control.groupby('customer_id', group_keys=False).apply(sample_orders).copy()
resampled_control['group_type'] = 'control_matched'

# 5️⃣ Combine test and control
comparison_df = pd.concat([viewed_test, resampled_control], ignore_index=True)

In [41]:
# 6️⃣ Validation checks
print("✅ Unique customer_id in test:   ", viewed_test['customer_id'].nunique())
print("✅ Unique customer_id in control:", resampled_control['customer_id'].nunique())
print("✅ Total orders in test:         ", len(viewed_test))
print("✅ Total orders in control:      ", len(resampled_control))

# 7️⃣ Compare the distribution of order counts per customer
test_dist = viewed_test.groupby('customer_id').size().value_counts().sort_index()
control_dist = resampled_control.groupby('customer_id').size().value_counts().sort_index()

# Plot
pd.DataFrame({'test': test_dist, 'control': control_dist}).fillna(0).plot.bar(
    title='Order Count per Customer Distribution',
    figsize=(10, 6)
)
plt.xlabel('Orders per Customer')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()


✅ Unique customer_id in test:    193156


KeyError: 'customer_id'

In [48]:
comparison_df.head(1)

,experiment_name,execution_name,sample_category,taxi_ltr_segment,taxi_retention_segments,taxi_regularity_segment,taxi_income_segment,taxi_intent_segment,yyyymmdd,customer_id,city_name,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,...,time_bucket,city,dispatch_viewed_cux,dispatch_clicked_cux,otw_viewed_cux,otw_clicked_cux,both_viewed_cux,frequency,gross_orders,accepted_orders,aor,mp_tag,fm_bin,lm_bin,dispatch_viewed_cux_tag,dispatch_clicked_cux_tag,otw_viewed_cux_tag,otw_clicked_cux_tag,both_viewed_cux_tag,hex_8_route,clicked,both_view,single_view,updated_segment,group_type
0,loyalty_experiment2,loyalty_experiment2_delhi,test,PHH,GOLD,WEEKLY,MEDIUM_INCOME,LOW,20250522,6054c22bba88aae23364c52c,Delhi,Link,682f11926946f36437a71d7c,dropped,NaN,1747915154756,1.747915e+12,NaN,NaN,NaN,143.0,1747915154756,1715,172914,9.86,...,evening_peak,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2976.0,2546.0,85.55,1. Good,0-500m,9+km,test,test,test,test,test,883da11637fffff - 883da114ebfffff,None,None,None,3. viewed,test_viewed


In [49]:
comparison_df.sample_category.value_counts(normalize=True)

sample_category
test       0.5
control    0.5
Name: proportion, dtype: float64

In [50]:
comparison_df.group_type.value_counts(normalize=True)

group_type
test_viewed        0.5
control_matched    0.5
Name: proportion, dtype: float64

In [51]:
group_type = ['control_matched', 'test_viewed']

for group_tc in group_type:
    print(group_tc)
    print(comparison_df.query(f"group_type == '{group_tc}'").taxi_regularity_segment.value_counts(normalize=True))

control_matched
taxi_regularity_segment
DAILY        0.395692
WEEKLY       0.238493
BI_WEEKLY    0.154559
UNKNOWN      0.077890
MONTHLY      0.067701
RARE_NEED    0.065665
Name: proportion, dtype: float64
test_viewed
taxi_regularity_segment
DAILY        0.395670
WEEKLY       0.238497
BI_WEEKLY    0.154558
UNKNOWN      0.077882
MONTHLY      0.067715
RARE_NEED    0.065677
Name: proportion, dtype: float64


In [52]:
comparison_df.groupby(['sample_category', 'group_type']).agg(cux=('customer_id', 'nunique'))

,,cux
sample_category,group_type,
control,control_matched,316970
test,test_viewed,143676


In [53]:
comparison_df.groupby(['sample_category', 'group_type']).agg(cux=('order_id', 'nunique'))

,,cux
sample_category,group_type,
control,control_matched,387498
test,test_viewed,387501


In [54]:
df_city_group_tc = get_funnel(comparison_df, ['city_name','sample_category'])
df_city_group_tc_dispatch_view = get_funnel(comparison_df, ['city_name','sample_category', 'dispatch_viewed_cux_tag'])
df_city_group_tc_otw_view = get_funnel(comparison_df, ['city_name','sample_category', 'otw_viewed_cux_tag'])
df_city_group_tc_both_view = get_funnel(comparison_df, ['city_name','sample_category', 'both_viewed_cux_tag'])
df_city_group_tc_updated_seg = get_funnel(comparison_df, ['city_name','sample_category', 'updated_segment'])

Execution time: 2.401 seconds
Execution time: 2.222 seconds
Execution time: 2.185 seconds
Execution time: 2.220 seconds
Execution time: 2.459 seconds


In [55]:
display(df_city_group_tc)
display(df_city_group_tc_dispatch_view)
display(df_city_group_tc_otw_view)
display(df_city_group_tc_both_view)
display(df_city_group_tc_updated_seg)

,city_name,sample_category,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,316970,387498,205286,52.98,20.93,18.49,0.92,6.68,129.4,268.1,71.4,53.0,104.0,182.0,272.0,51.0,156.0,361.0,633.0,6.0,11.0,60.0,177.9
1,Delhi,test,143676,387501,204713,52.83,20.71,18.65,0.98,6.83,121.1,263.7,67.8,47.0,96.0,171.0,262.0,46.0,146.0,349.0,624.0,6.0,11.0,58.0,167.0


,city_name,sample_category,dispatch_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,316970,387498,205286,52.98,20.93,18.49,0.92,6.68,129.4,268.1,71.4,53.0,104.0,182.0,272.0,51.0,156.0,361.0,633.0,6.0,11.0,60.0,177.9
1,Delhi,test,test,143676,387501,204713,52.83,20.71,18.65,0.98,6.83,121.1,263.7,67.8,47.0,96.0,171.0,262.0,46.0,146.0,349.0,624.0,6.0,11.0,58.0,167.0


,city_name,sample_category,otw_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,316970,387498,205286,52.98,20.93,18.49,0.92,6.68,129.4,268.1,71.4,53.0,104.0,182.0,272.0,51.0,156.0,361.0,633.0,6.0,11.0,60.0,177.9
1,Delhi,test,test,143676,387501,204713,52.83,20.71,18.65,0.98,6.83,121.1,263.7,67.8,47.0,96.0,171.0,262.0,46.0,146.0,349.0,624.0,6.0,11.0,58.0,167.0


,city_name,sample_category,both_viewed_cux_tag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,control,316970,387498,205286,52.98,20.93,18.49,0.92,6.68,129.4,268.1,71.4,53.0,104.0,182.0,272.0,51.0,156.0,361.0,633.0,6.0,11.0,60.0,177.9
1,Delhi,test,test,143676,387501,204713,52.83,20.71,18.65,0.98,6.83,121.1,263.7,67.8,47.0,96.0,171.0,262.0,46.0,146.0,349.0,624.0,6.0,11.0,58.0,167.0


,city_name,sample_category,updated_segment,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrm %,expired %,cobra_ttc_mean,ocara_ttc_mean,tta_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,tta_p25,tta_p50,tta_p75,tta_p90
0,Delhi,control,1. control,316970,387498,205286,52.98,20.93,18.49,0.92,6.68,129.4,268.1,71.4,53.0,104.0,182.0,272.0,51.0,156.0,361.0,633.0,6.0,11.0,60.0,177.9
1,Delhi,test,3. viewed,143676,387501,204713,52.83,20.71,18.65,0.98,6.83,121.1,263.7,67.8,47.0,96.0,171.0,262.0,46.0,146.0,349.0,624.0,6.0,11.0,58.0,167.0


In [56]:
comparison_df.to_parquet('may_28_v0.parquet')